# Multi-Disease Chest X-Ray Classification Using Adult and Pediatric Data with Explainable AI

**Proposed system:** *Age-Aware Gated-Fusion Hybrid (AGF-Net)*, a **lightweight** (about 9.5 M parameters) EfficientNet-B0 + MobileViTv2 sample-wise gated fusion network. **One model** is trained **jointly** on adult and pediatric chest X-rays and predicts 14 diseases for both, with:

1. **Leakage control:** patient-grouped splits, perceptual-hash near-duplicate grouping (within and across datasets), and a locked test set evaluated once.
2. **Partial-label learning:** a unified 14-disease label space; labels a dataset never annotated are *masked*, not treated as negative.
3. **Age awareness:** an auxiliary age-group head, FiLM conditioning on age group (known age or predicted), and **age-specific prevalence (logit) adjustment** so the network does not learn "child-looking → pneumonia" shortcuts.
4. **Age-specific calibration:** per-label, per-age-group temperature scaling and decision thresholds fixed on validation data only.
5. **Explainability with numbers:** gate-fused Grad-CAM / HiResCAM, pointing game and energy gain on NIH expert bounding boxes, deletion faithfulness, border-saliency shortcut audit.
6. **Anti-shortcut training (proposed `joint_age_adv`):** a hospital discriminator behind a gradient-reversal layer, plus source-balanced pediatric sampling, so the features cannot simply recognise which hospital a child's X-ray came from.
7. **Rigorous evaluation:** separate adult and pediatric test sets, patient-cluster bootstrap 95% CIs, age-band/sex subgroup analysis (underdiagnosis rate), robustness to corruptions, paired comparison between experiments.

### Datasets (add both in Kaggle with **+ Add Input**)
| Role | Dataset | Kaggle slug |
|---|---|---|
| Adult (and some children), 14 diseases, bounding boxes | NIH ChestX-ray14 | `nih-chest-xrays/data` |
| Pediatric, ages 1–5, Normal / Pneumonia (bacterial, viral) | Guangzhou pediatric CXR (Kermany et al.) | `paultimothymooney/chest-xray-pneumonia` |
| *Optional* external pediatric multi-disease | VinDr-PCXR (PhysioNet, credentialed) | upload privately, then set `EXTERNAL_CSV` |

### How to run
* Accelerator: **GPU T4 ×1 or P100**. Internet: **On** (to download ImageNet weights).
* Run cells top to bottom. One session trains **one experiment** (`CFG['EXPERIMENT']`). Results are saved to `/kaggle/working/results/<experiment>/`.
* Suggested order (one Kaggle session each): `joint_age_adv` (proposed) → `joint_age` → `adult_only` → `joint` → `joint_naive` → `ped_only`. Then download/attach previous result folders and run the final **comparison** section.
* First run: set `DEBUG=True` to check that everything works in ~10 minutes, then set it back to `False`.

## 0. Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['timm']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

import os, re, gc, json, math, time, glob, random, warnings, copy
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings('ignore')
cv2.setNumThreads(0)
print('torch', torch.__version__, '| timm', timm.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
CFG = dict(
    EXPERIMENT='joint_age_adv',  # 'joint_age_adv' (proposed) | 'joint_age' | 'joint' | 'joint_naive' | 'adult_only' | 'ped_only'
    INPUT_ROOT='/kaggle/input',
    WORK_DIR='/kaggle/working',
    CACHE_DIR=None,              # None -> WORK_DIR/cache. Point to an attached dataset to reuse a previous cache.
    CACHE_SIZE=256,              # cached square size (all images resized the same way: no letterbox borders)
    IMG_SIZE=256,
    USE_CLAHE=False,             # ablation switch
    # Lightweight hybrid (~9.5 M parameters in total): EfficientNet-B0 (CNN) + MobileViTv2-1.0 (CNN + transformer).
    # Heavier option: ['convnext_tiny.fb_in22k_ft_in1k', 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k'] with IMG_SIZE=224.
    BACKBONES=['efficientnet_b0.ra_in1k', 'mobilevitv2_100.cvnets_in1k'],
    PRETRAINED=True,
    PROJ_DIM=512, DROPOUT=0.3,
    BATCH_SIZE=64, NUM_WORKERS=min(8, os.cpu_count() or 2),
    EPOCHS=15, FREEZE_EPOCHS=1, EPOCH_SAMPLES=60000,  # images drawn per epoch (domain-balanced sampler)
    LR_BACKBONE=3e-4, LR_HEAD=1e-3, WD=0.05, WARMUP_FRAC=0.05,
    EMA_DECAY=0.999, PATIENCE=4, GRAD_CLIP=1.0, GRAD_CHECKPOINT=False,
    PED_FRAC=0.30,               # share of pediatric images in each training epoch (joint experiments)
    AGE_DROP=0.30,               # prob. of hiding the known age group during training (model must also work without age)
    LAMBDA_AGE=0.2, LAMBDA_SUB=0.3, LOGIT_ADJ_TAU=1.0,
    LAMBDA_ADV=0.3,              # weight of the hospital-adversarial loss (joint_age_adv only)
    ADULT_TRAIN_FRACTION=1.0,    # subsample NIH training patients for faster runs
    VAL_FOLDS=10, SEED=42,
    KERMANY_SPLIT='auto',        # 'auto': keep official test unless patient/duplicate leakage is found; 'regroup': always regroup
    PHASH_MAX_HAMMING=3, PHASH_CROSS=True, DUP_MIN_CORR=0.95,   # candidate pairs must also match on 64x64 thumbnails
    N_BOOT=500,
    INFER_USE_KNOWN_AGE=True,
    EXTERNAL_CSV=None,           # optional CSV: path, patient_id, age, sex, + any of the 14 label columns (0/1, blank=unknown)
    MAX_TRAIN_HOURS=8.0,
    LIME_IMAGES=30, LIME_SAMPLES=1000, SHAP_SAMPLES=1000, N_SUPERPIXELS=80,  # LIME / KernelSHAP
    CAM_SMOOTH=8,                # Grad-CAM/HiResCAM averaged over 8 slightly noisy copies (SmoothGrad-style)
    CAM_MAX_BATCH=16,            # images x copies with gradients per chunk (lower it if Grad-CAM runs out of memory)
    FEWSHOT_TARGETS=['Pediatric-GWCMC', 'NIH-pediatric', 'External'], FEWSHOT_KS=[1, 2, 5, 10, 20, 50], FEWSHOT_REPS=5,
    LAUNCH_WEBAPP=False,         # True -> installs gradio and opens the web app with a public share link
    DEBUG=False,                 # tiny subsample, 1 epoch: smoke test
)
# optional override (used for local testing): export CXR_CFG_OVERRIDE='{"DEBUG": true}'
CFG.update(json.loads(os.environ.get('CXR_CFG_OVERRIDE', '{}')))
if CFG['DEBUG']:
    CFG.update(EPOCHS=max(1, min(CFG['EPOCHS'], 2)), FREEZE_EPOCHS=0, EPOCH_SAMPLES=min(CFG['EPOCH_SAMPLES'], 1500), N_BOOT=50,
               LIME_IMAGES=4, LIME_SAMPLES=64, SHAP_SAMPLES=64, FEWSHOT_REPS=2, CAM_SMOOTH=2)

EXP = CFG['EXPERIMENT']
FLAGS = {
    'adult_only':  dict(use_adult=True,  use_ped=False, mask=True,  film=False, age_head=False, logit_adj=False, sub_head=False),
    'ped_only':    dict(use_adult=False, use_ped=True,  mask=True,  film=False, age_head=False, logit_adj=False, sub_head=True),
    'joint_naive': dict(use_adult=True,  use_ped=True,  mask=False, film=False, age_head=False, logit_adj=False, sub_head=True),
    'joint':       dict(use_adult=True,  use_ped=True,  mask=True,  film=False, age_head=False, logit_adj=False, sub_head=True),
    'joint_age':   dict(use_adult=True,  use_ped=True,  mask=True,  film=True,  age_head=True,  logit_adj=True,  sub_head=True),
    # proposed: joint_age + anti-shortcut package (hospital-adversarial head with gradient reversal + source-balanced pediatric sampling)
    'joint_age_adv': dict(use_adult=True, use_ped=True, mask=True, film=True, age_head=True, logit_adj=True, sub_head=True),
}[EXP]
FLAGS = dict(FLAGS, adv=(EXP == 'joint_age_adv'), src_balance=(EXP == 'joint_age_adv'))

WORK = Path(CFG['WORK_DIR']); WORK.mkdir(parents=True, exist_ok=True)
CACHE = Path(CFG['CACHE_DIR']) if CFG['CACHE_DIR'] else WORK / 'cache'
OUT = WORK / 'results' / EXP; OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / 'figures'; FIG.mkdir(exist_ok=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG['SEED'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'

LABELS = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia',
          'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']
NL = len(LABELS); PNEU = LABELS.index('Pneumonia')
json.dump(CFG, open(OUT / 'config.json', 'w'), indent=2)
print('Experiment:', EXP, FLAGS, '\nDevice:', DEVICE, '| output ->', OUT)

## 1. Locate the datasets
The cell walks `/kaggle/input` once, so it works whatever folder name Kaggle gives the datasets.

In [ ]:
t0 = time.time()
nih_csv = bbox_csv = tv_list = te_list = None
nih_imgs, ker_imgs = {}, {}
NIH_RE = re.compile(r'^\d{8}_\d{3}\.png$')
for root, dirs, files in os.walk(CFG['INPUT_ROOT']):
    if '__MACOSX' in root:
        continue
    for f in files:
        if re.match(r'Data_Entry_2017.*\.csv$', f) and nih_csv is None: nih_csv = os.path.join(root, f)
        elif re.match(r'(?i)bbox_list_2017.*\.csv$', f): bbox_csv = os.path.join(root, f)
        elif f == 'train_val_list.txt': tv_list = os.path.join(root, f)
        elif f == 'test_list.txt': te_list = os.path.join(root, f)
        elif NIH_RE.match(f): nih_imgs.setdefault(f, os.path.join(root, f))
        elif f.lower().endswith(('.jpeg', '.jpg', '.png')):
            parts = Path(root).parts
            if len(parts) >= 3 and parts[-1] in ('NORMAL', 'PNEUMONIA') and parts[-2] in ('train', 'val', 'test'):
                key = f'{parts[-2]}/{parts[-1]}/{f}'      # dedupe the duplicated chest_xray/chest_xray folder
                ker_imgs.setdefault(key, os.path.join(root, f))
print(f'NIH csv: {nih_csv}\nNIH bbox: {bbox_csv}\nNIH lists: {tv_list}, {te_list}')
print(f'NIH images found: {len(nih_imgs):,} | Pediatric (Kermany) images found: {len(ker_imgs):,} | {time.time()-t0:.0f}s')
assert nih_csv is not None and len(nih_imgs) > 0, 'Add the NIH ChestX-ray14 dataset (nih-chest-xrays/data)'
assert len(ker_imgs) > 0, 'Add the pediatric pneumonia dataset (paultimothymooney/chest-xray-pneumonia)'

## 2. Build a unified metadata table
* **NIH:** 14 labels from `Finding Labels`; ages > 100 are recording errors and set to missing; patients < 18 years are the *NIH pediatric* subset.
* **Pediatric (Kermany):** NORMAL → all 14 labels negative; PNEUMONIA → `Pneumonia = 1` and the other 13 labels **unknown** (masked). Subtype bacterial / viral from the file name. Patient ID is parsed from the file name (`person123_…`, `IM-0115-…`).

In [ ]:
nih = pd.read_csv(nih_csv)
nih = nih[nih['Image Index'].isin(nih_imgs)].reset_index(drop=True)
age = pd.to_numeric(nih['Patient Age'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
age[(age > 100) | (age < 0)] = np.nan
test_names = set(open(te_list).read().split()) if te_list else set()
meta_nih = pd.DataFrame({
    'uid': 'NIH_' + nih['Image Index'], 'source': 'NIH', 'src_path': nih['Image Index'].map(nih_imgs),
    'patient_id': 'NIH_' + nih['Patient ID'].astype(str), 'age': age.values,
    'sex': nih['Patient Gender'].astype(str).str.upper().str[0], 'view': nih.get('View Position', pd.Series(['NA']*len(nih))).values,
    'official_split': np.where(nih['Image Index'].isin(test_names), 'test', 'trainval'), 'subtype': -1,
})
findings = nih['Finding Labels'].astype(str).str.split('|')
for L in LABELS:
    meta_nih[L] = findings.apply(lambda s: float(L in s)).values
if not test_names:   # fallback: patient-grouped 20% test
    gkf = GroupKFold(5); _, te_idx = next(gkf.split(meta_nih, groups=meta_nih.patient_id))
    meta_nih.loc[te_idx, 'official_split'] = 'test'

def ker_pid(fn):
    m = re.match(r'(person\d+)_', fn)
    if m: return 'KER_' + m.group(1)
    m = re.match(r'((?:NORMAL2-)?IM-\d+)', fn)
    return 'KER_' + (m.group(1) if m else fn)

rows = []
for key, p in sorted(ker_imgs.items()):
    sp, cls, fn = key.split('/')
    sub = 0 if 'bacteria' in fn.lower() else (1 if 'virus' in fn.lower() else -1)
    r = {'uid': 'KER_' + key.replace('/', '_'), 'source': 'KER', 'src_path': p, 'patient_id': ker_pid(fn),
         'age': np.nan, 'sex': 'U', 'view': 'AP', 'official_split': 'test' if sp == 'test' else 'trainval',
         'subtype': sub if cls == 'PNEUMONIA' else -1}
    for L in LABELS:
        r[L] = 0.0 if cls == 'NORMAL' else (1.0 if L == 'Pneumonia' else np.nan)
    rows.append(r)
meta_ker = pd.DataFrame(rows)

meta = pd.concat([meta_nih, meta_ker], ignore_index=True)
if CFG['EXTERNAL_CSV']:
    ext = pd.read_csv(CFG['EXTERNAL_CSV'])
    e = pd.DataFrame({'uid': 'EXT_' + ext.index.astype(str), 'source': 'EXT', 'src_path': ext['path'],
                      'patient_id': 'EXT_' + ext.get('patient_id', ext.index).astype(str),
                      'age': pd.to_numeric(ext.get('age', np.nan), errors='coerce'),
                      'sex': ext.get('sex', 'U'), 'view': 'NA', 'official_split': 'test', 'subtype': -1})
    for L in LABELS:
        e[L] = pd.to_numeric(ext[L], errors='coerce') if L in ext else np.nan
    meta = pd.concat([meta, e], ignore_index=True)

# age group: 1 = pediatric (<18 y), 0 = adult; Kermany children are 1-5 years old
meta['age_group'] = np.where(meta.source == 'KER', 1, np.where(meta.age < 18, 1, np.where(meta.age >= 18, 0, -1))).astype(int)
bins = [-1, 5, 12, 17, 40, 60, 80, 200]; names = ['0-5', '6-12', '13-17', '18-40', '41-60', '61-80', '81+']
meta['age_band'] = pd.cut(meta.age, bins=bins, labels=names).astype(object).fillna('nan').astype(str)
meta.loc[meta.source == 'KER', 'age_band'] = '0-5'

if CFG['DEBUG']:
    keep_p = []
    for src, n in [('NIH', 600), ('KER', 300)]:
        pids = meta[meta.source == src].patient_id.unique()
        kids = meta[(meta.source == src) & (meta.age_group == 1)].patient_id.unique()
        rs = np.random.RandomState(0)
        keep_p += list(rs.choice(pids, min(n, len(pids)), replace=False)) + list(kids[:100])
    meta = meta[meta.patient_id.isin(set(keep_p)) | (meta.source == 'EXT')].reset_index(drop=True)

print(meta.groupby(['source', 'official_split', 'age_group']).size().rename('images').to_frame())
print('\nPatients:', meta.groupby('source').patient_id.nunique().to_dict())

### 2.1 Data audit: age distribution, label prevalence by age group

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
meta[meta.source == 'NIH'].age.dropna().plot.hist(bins=50, ax=ax[0], color='#4C72B0'); ax[0].axvline(18, c='r', ls='--')
ax[0].set_title('NIH patient age (red: 18 y)'); ax[0].set_xlabel('age (years)')
meta.age_band.value_counts().reindex(names).plot.bar(ax=ax[1], color='#55A868'); ax[1].set_title('Images per age band (all sources)')
prev = pd.DataFrame({g: meta[(meta.age_group == k)][LABELS].mean() for k, g in [(0, 'adult'), (1, 'pediatric')]})
prev.plot.barh(ax=ax[2]); ax[2].set_title('Label prevalence (known labels only)')
plt.tight_layout(); plt.savefig(FIG / 'data_audit.png', dpi=120); plt.show()
print('NIH pediatric images (<18 y):', int(((meta.source == 'NIH') & (meta.age_group == 1)).sum()))
print('Kermany classes:', meta_ker.assign(c=np.where(meta_ker.Pneumonia == 1, 'PNEUMONIA', 'NORMAL')).groupby(['official_split', 'c']).size().to_dict())

## 3. Preprocess once and cache (+ perceptual hash)
Every image is read as grayscale, optionally CLAHE-equalised, and resized to a square **without letterbox padding**: black padding would appear only on the non-square pediatric images and give the model a dataset-source shortcut. A 64-bit DCT perceptual hash is computed in the same pass for near-duplicate detection.

In [ ]:
CACHE.mkdir(parents=True, exist_ok=True)
S = CFG['CACHE_SIZE']
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def phash64(img):
    small = cv2.resize(img, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    d = cv2.dct(small)[:8, :8].flatten()
    bits = (d > np.median(d[1:])).astype(np.uint64)
    h = int((bits << np.arange(64, dtype=np.uint64)).sum())
    return h - (1 << 64) if h >= (1 << 63) else h   # store as signed int64

def preprocess_one(args):
    uid, src = args
    dst = CACHE / f'{uid}.jpg'
    img = cv2.imread(str(dst), cv2.IMREAD_GRAYSCALE) if dst.exists() else None
    h = w = -1
    if img is not None:
        try:
            from PIL import Image
            with Image.open(src) as im: w, h = im.size
        except Exception:
            pass
    if img is None:
        raw = cv2.imread(src, cv2.IMREAD_GRAYSCALE)
        if raw is None:
            return uid, None, -1, -1, 0
        h, w = raw.shape[:2]
        if CFG['USE_CLAHE']:
            raw = _clahe.apply(raw)
        img = cv2.resize(raw, (S, S), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(dst), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    return uid, str(dst), h, w, phash64(img)

meta_cache_f = CACHE / 'meta_cache.csv'
old = pd.read_csv(meta_cache_f) if meta_cache_f.exists() else pd.DataFrame(columns=['uid', 'path', 'orig_h', 'orig_w', 'phash'])
old = old.set_index('uid')
todo = [(u, s) for u, s in zip(meta.uid, meta.src_path) if u not in old.index]
print(f'{len(todo):,} images to preprocess ({len(old):,} already cached)')
t0 = time.time(); res = []
with ThreadPoolExecutor(max_workers=max(4, (os.cpu_count() or 2) * 2)) as ex:
    for i, r in enumerate(ex.map(preprocess_one, todo, chunksize=64)):
        res.append(r)
        if (i + 1) % 10000 == 0:
            print(f'  {i+1:,}/{len(todo):,}  {time.time()-t0:.0f}s')
new = pd.DataFrame(res, columns=['uid', 'path', 'orig_h', 'orig_w', 'phash']).set_index('uid')
cache_tab = pd.concat([old, new]); cache_tab = cache_tab[~cache_tab.index.duplicated(keep='last')]
try:
    cache_tab.reset_index().to_csv(meta_cache_f, index=False)
except OSError:
    pass   # read-only cache directory
meta = meta.join(cache_tab, on='uid')
bad = meta.path.isna().sum()
meta = meta[meta.path.notna()].reset_index(drop=True)
meta['phash'] = meta.phash.astype(np.int64)
print(f'Unreadable images removed: {bad} | cached: {len(meta):,} | {time.time()-t0:.0f}s')

## 4. Leakage control and splits
1. Patient overlap between official train and test lists (NIH, Kermany).
2. Near-duplicates (pHash Hamming ≤ `PHASH_MAX_HAMMING`) inside the pediatric set and **across** adult/pediatric datasets; near-duplicates are merged with a union–find into one group.
3. Splits are grouped by patient **and** duplicate group: NIH uses the official patient-wise test list as the locked test; validation is a stratified grouped fold of train-val. The pediatric official test is kept only if no group crosses it (`KERMANY_SPLIT='auto'`).

In [ ]:
def popcount64(x):
    x = x - ((x >> np.uint64(1)) & np.uint64(0x5555555555555555))
    x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0F0F0F0F0F0F0F0F)
    return ((x * np.uint64(0x0101010101010101)) >> np.uint64(56)).astype(np.int32)

def near_dup_pairs(ha, hb, thr, same=False, chunk=2048):
    pairs = []
    for s in range(0, len(ha), chunk):
        d = popcount64(ha[s:s+chunk, None] ^ hb[None, :])
        i, j = np.nonzero(d <= thr); i = i + s
        if same:
            k = i < j; i, j = i[k], j[k]
        pairs.append(np.stack([i, j], 1))
    return np.concatenate(pairs) if pairs else np.zeros((0, 2), int)

_thumbs = {}
def thumb(j):
    if j not in _thumbs:
        t = cv2.resize(cv2.imread(meta.path.values[j], cv2.IMREAD_GRAYSCALE), (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
        _thumbs[j] = (t - t.mean()) / (t.std() + 1e-6)
    return _thumbs[j]

def verify(pairs_global):
    # pHash is only a candidate filter: chest X-rays look alike at low frequency, so confirm with thumbnail correlation
    ok = [(a, b) for a, b in pairs_global if float(np.dot(thumb(a), thumb(b)) / thumb(a).size) >= CFG['DUP_MIN_CORR']]
    return np.array(ok, dtype=int).reshape(-1, 2)

class UF:
    def __init__(s, n): s.p = np.arange(n)
    def f(s, a):
        while s.p[a] != a:
            s.p[a] = s.p[s.p[a]]; a = s.p[a]
        return a
    def u(s, a, b):
        a, b = s.f(a), s.f(b)
        if a != b: s.p[b] = a

thr = CFG['PHASH_MAX_HAMMING']
H = meta.phash.values.astype(np.int64).view(np.uint64)
uf = UF(len(meta))
# group by patient
for _, idx in meta.groupby('patient_id').indices.items():
    for k in idx[1:]: uf.u(idx[0], k)
# near-duplicates inside pediatric + external
ped_idx = np.where(meta.source.isin(['KER', 'EXT']).values)[0]
cand = near_dup_pairs(H[ped_idx], H[ped_idx], thr, same=True)
pp = verify(np.stack([ped_idx[cand[:, 0]], ped_idx[cand[:, 1]]], 1)) if len(cand) else np.zeros((0, 2), int)
for i, j in pp: uf.u(i, j)
print(f'Pediatric near-duplicates: {len(cand)} pHash candidates -> {len(pp)} confirmed')
# across datasets (adult NIH vs pediatric)
cross = np.zeros((0, 2), int)
if CFG['PHASH_CROSS']:
    nih_idx = np.where(meta.source.values == 'NIH')[0]
    cand = near_dup_pairs(H[nih_idx], H[ped_idx], thr)
    cross = verify(np.stack([nih_idx[cand[:, 0]], ped_idx[cand[:, 1]]], 1)) if len(cand) else np.zeros((0, 2), int)
    for i, j in cross: uf.u(i, j)
    print(f'Adult-pediatric cross-dataset near-duplicates: {len(cand)} candidates -> {len(cross)} confirmed')
meta['group'] = [f'G{uf.f(i)}' for i in range(len(meta))]
gs = meta.groupby('group').size()
print(f'Groups: {len(gs):,} | largest group: {gs.max()} images (a very large group means the duplicate threshold is too loose)')

# patient overlap audit on official splits
for src in ['NIH', 'KER']:
    m = meta[meta.source == src]
    ov = set(m[m.official_split == 'trainval'].group) & set(m[m.official_split == 'test'].group)
    print(f'{src}: groups (patient + near-dup) crossing official train/test: {len(ov)}')

# --- final splits ---
meta['split'] = 'train'
rs = CFG['SEED']
anypos = (meta[LABELS].fillna(0).values.sum(1) > 0).astype(int)
# NIH: official test locked; validation fold grouped & stratified on (pediatric, any finding)
m_nih = (meta.source == 'NIH').values
meta.loc[m_nih & (meta.official_split == 'test').values, 'split'] = 'test'
tv = np.where(m_nih & (meta.official_split == 'trainval').values)[0]
strat = meta.age_group.values[tv].clip(0) * 2 + anypos[tv]
_, va = next(StratifiedGroupKFold(CFG['VAL_FOLDS'], shuffle=True, random_state=rs).split(tv, strat, meta.group.values[tv]))
meta.loc[tv[va], 'split'] = 'val'
# NIH groups that cross the official test list (should be none) are pushed to test
bad_g = set(meta[m_nih & (meta.split == 'test')].group) & set(meta[m_nih & (meta.split != 'test')].group)
meta.loc[m_nih & meta.group.isin(bad_g).values, 'split'] = 'test'
# Pediatric
m_ker = (meta.source == 'KER').values
ker_cross = set(meta[m_ker & (meta.official_split == 'test')].group) & set(meta[m_ker & (meta.official_split == 'trainval')].group)
regroup = CFG['KERMANY_SPLIT'] == 'regroup' or (CFG['KERMANY_SPLIT'] == 'auto' and len(ker_cross) > 0)
ki = np.where(m_ker)[0]
ycls = (meta.Pneumonia.values[ki] == 1).astype(int) + (meta.subtype.values[ki] == 1).astype(int)
if regroup:
    print(f'Pediatric official split leaks through {len(ker_cross)} groups -> regrouping (patient + duplicate groups)')
    _, te = next(StratifiedGroupKFold(8, shuffle=True, random_state=rs).split(ki, ycls, meta.group.values[ki]))
    meta.loc[ki[te], 'split'] = 'test'
else:
    meta.loc[m_ker & (meta.official_split == 'test').values, 'split'] = 'test'
ktv = np.where(m_ker & (meta.split != 'test').values)[0]
_, va = next(StratifiedGroupKFold(10, shuffle=True, random_state=rs).split(ktv, ycls[np.searchsorted(ki, ktv)], meta.group.values[ktv]))
meta.loc[ktv[va], 'split'] = 'val'
meta.loc[meta.source == 'EXT', 'split'] = 'test'
# Any group (patient / near-duplicate, possibly across datasets) that still spans several splits:
# keep its test members, drop its train/val members ('excluded'); then train-vs-val conflicts -> val
for g, sp in meta[meta.split.isin(['train', 'val', 'test'])].groupby('group').split:
    u = set(sp)
    if len(u) > 1:
        keep = 'test' if 'test' in u else 'val'
        meta.loc[(meta.group == g) & (meta.split != keep) & meta.split.isin(['train', 'val', 'test']), 'split'] = 'excluded'
print('Images excluded because they duplicate an image in another split:', int((meta.split == 'excluded').sum()))

# final leakage assertion
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    inter = set(meta[meta.split == a].group) & set(meta[meta.split == b].group)
    assert len(inter) == 0, f'leakage between {a} and {b}: {len(inter)} groups'
print('No patient or near-duplicate group is shared between train / val / test.')

# adult training subsample (by patient)
if CFG['ADULT_TRAIN_FRACTION'] < 1:
    tr_p = meta[(meta.source == 'NIH') & (meta.split == 'train')].patient_id.unique()
    keep = set(np.random.RandomState(rs).choice(tr_p, int(len(tr_p) * CFG['ADULT_TRAIN_FRACTION']), replace=False))
    meta.loc[(meta.source == 'NIH') & (meta.split == 'train') & ~meta.patient_id.isin(keep), 'split'] = 'unused'

meta['test_set'] = np.select(
    [(meta.split == 'test') & (meta.source == 'NIH') & (meta.age_group == 0),
     (meta.split == 'test') & (meta.source == 'NIH') & (meta.age_group == 1),
     (meta.split == 'test') & (meta.source == 'KER'),
     (meta.split == 'test') & (meta.source == 'EXT')],
    ['NIH-adult', 'NIH-pediatric', 'Pediatric-GWCMC', 'External'], default='')
print(meta.groupby(['split', 'source', 'age_group']).size().unstack(fill_value=0))
meta.drop(columns=['src_path']).to_csv(OUT / 'meta_splits.csv', index=False)

## 5. Label matrix, masks, priors and experiment filter
* `mask=0` for labels a dataset never annotated (partial labels). The **joint_naive** ablation fills them with 0 (the common but wrong practice).
* Class weights: `pos_weight = sqrt(neg/pos)` per label on the training split.
* Age-group priors π(g, c) feed the logit adjustment of the proposed model.

In [ ]:
Yraw = meta[LABELS].values.astype(np.float32)
MASK = (~np.isnan(Yraw)).astype(np.float32)
if not FLAGS['mask']:
    MASK = np.ones_like(MASK)
Y = np.nan_to_num(Yraw, nan=0.0).astype(np.float32)
AGEG = meta.age_group.values.astype(np.float32)

in_domain = np.ones(len(meta), bool)
if not FLAGS['use_adult']: in_domain &= (meta.age_group.values == 1)
if not FLAGS['use_ped']:   in_domain &= (meta.age_group.values == 0)
tr_idx = np.where((meta.split.values == 'train') & in_domain)[0]
va_idx = np.where((meta.split.values == 'val') & in_domain)[0]
te_idx = np.where(meta.split.values == 'test')[0]
print(f'train {len(tr_idx):,} | val {len(va_idx):,} | test {len(te_idx):,}')

pos = (Y[tr_idx] * MASK[tr_idx]).sum(0); tot = MASK[tr_idx].sum(0)
POS_WEIGHT = torch.tensor(np.clip(np.sqrt((tot - pos) / np.maximum(pos, 1)), 1, 30), dtype=torch.float32)
PRIORS = np.zeros((2, NL), np.float32)
p_pool = pos / np.maximum(tot, 1)
for g in (0, 1):
    s = tr_idx[AGEG[tr_idx] == g]
    n_known = MASK[s].sum(0) if len(s) else np.zeros(NL)
    p = (Y[s] * MASK[s]).sum(0) / np.maximum(n_known, 1) if len(s) else p_pool
    PRIORS[g] = np.clip(np.where(n_known >= 20, p, p_pool), 1e-3, 0.999)   # too few known labels -> pooled prior
PRIOR_LOGIT = np.log(PRIORS / (1 - PRIORS))
print(pd.DataFrame({'pos_weight': POS_WEIGHT.numpy(), 'prior_adult': PRIORS[0], 'prior_ped': PRIORS[1],
                    'train_pos': pos.astype(int)}, index=LABELS).round(3))

## 6. Dataset, augmentation and domain-balanced sampler

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], np.float32)[:, None, None]; STD = np.array([0.229, 0.224, 0.225], np.float32)[:, None, None]
ISZ = CFG['IMG_SIZE']

def augment(img, rng):
    h, w = img.shape
    sc, ar = rng.uniform(0.80, 1.0), rng.uniform(0.9, 1.1)
    ch, cw = min(h, int(h * math.sqrt(sc / ar))), min(w, int(w * math.sqrt(sc * ar)))
    y0, x0 = rng.integers(0, h - ch + 1), rng.integers(0, w - cw + 1)
    img = cv2.resize(img[y0:y0 + ch, x0:x0 + cw], (ISZ, ISZ), interpolation=cv2.INTER_AREA)
    M = cv2.getRotationMatrix2D((ISZ / 2, ISZ / 2), rng.uniform(-10, 10), rng.uniform(0.95, 1.05))
    M[:, 2] += rng.uniform(-0.04, 0.04, 2) * ISZ
    img = cv2.warpAffine(img, M, (ISZ, ISZ), borderMode=cv2.BORDER_REFLECT_101)   # no black borders
    f = img.astype(np.float32) / 255.0
    f = np.clip(f * rng.uniform(0.85, 1.15) + rng.uniform(-0.08, 0.08), 0, 1) ** rng.uniform(0.8, 1.25)
    if rng.random() < 0.2:
        f = np.clip(f + rng.normal(0, 0.02, f.shape).astype(np.float32), 0, 1)
    return f   # no horizontal flip: it would move the heart to the right side

def corrupt(f, kind, sev):
    if kind == 'noise':    return np.clip(f + np.random.normal(0, [0.03, 0.06, 0.10][sev], f.shape).astype(np.float32), 0, 1)
    if kind == 'blur':     return cv2.GaussianBlur(f, (0, 0), [1.0, 2.0, 3.0][sev])
    if kind == 'contrast': c = [0.75, 0.55, 0.35][sev]; return np.clip((f - f.mean()) * c + f.mean(), 0, 1)
    if kind == 'jpeg':
        ok, b = cv2.imencode('.jpg', (f * 255).astype(np.uint8), [cv2.IMWRITE_JPEG_QUALITY, [30, 15, 8][sev]])
        return cv2.imdecode(b, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255
    return f

class CXRDataset(Dataset):
    def __init__(self, idx, train=False, corruption=None):
        self.idx, self.train, self.corruption = np.asarray(idx), train, corruption
        self.paths = meta.path.values
    def __len__(self): return len(self.idx)
    def load(self, j):
        img = cv2.imread(self.paths[j], cv2.IMREAD_GRAYSCALE)
        if self.train:
            f = augment(img, np.random.default_rng((torch.initial_seed() + int(j)) % 2**32))
        else:
            f = cv2.resize(img, (ISZ, ISZ), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
        if self.corruption is not None:
            f = corrupt(f, *self.corruption)
        return ((np.repeat(f[None], 3, 0) - MEAN) / STD).astype(np.float32)
    def __getitem__(self, k):
        j = self.idx[k]
        ag = AGEG[j]
        return dict(x=torch.from_numpy(self.load(j)), y=torch.from_numpy(Y[j]), m=torch.from_numpy(MASK[j]),
                    age=torch.tensor(max(ag, 0.0)), age_known=torch.tensor(float(ag >= 0)),
                    sub=torch.tensor(int(meta.subtype.values[j]) if FLAGS['sub_head'] else -1),
                    src=torch.tensor(float(SRC_KER[j])), j=torch.tensor(j))

SRC_KER = (meta.source.values == 'KER').astype(np.float32)   # 1 = Guangzhou pediatric hospital, 0 = NIH

def make_train_loader():
    g = AGEG[tr_idx]
    w = np.ones(len(tr_idx))
    if FLAGS['use_adult'] and FLAGS['use_ped'] and (g == 1).any() and (g != 1).any():
        w[g == 1] = CFG['PED_FRAC'] / (g == 1).sum()
        w[g != 1] = (1 - CFG['PED_FRAC']) / (g != 1).sum()
        sk = SRC_KER[tr_idx]
        if FLAGS['src_balance'] and ((g == 1) & (sk == 1)).any() and ((g == 1) & (sk == 0)).any():
            # children: half of the pediatric share from NIH children, half from Guangzhou
            for v in (0, 1):
                kk = (g == 1) & (sk == v)
                w[kk] = CFG['PED_FRAC'] / 2 / kk.sum()
    n = min(CFG['EPOCH_SAMPLES'], max(len(tr_idx), 1) * 3)
    sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), n, replacement=True)
    return DataLoader(CXRDataset(tr_idx, train=True), batch_size=CFG['BATCH_SIZE'], sampler=sampler,
                      num_workers=CFG['NUM_WORKERS'], pin_memory=AMP, drop_last=True, persistent_workers=CFG['NUM_WORKERS'] > 0)

def make_loader(idx, corruption=None, bs=None):
    return DataLoader(CXRDataset(idx, corruption=corruption), batch_size=bs or CFG['BATCH_SIZE'] * 2, shuffle=False,
                      num_workers=CFG['NUM_WORKERS'], pin_memory=AMP)

b = next(iter(make_loader(tr_idx[:8], bs=8)))
print(b['x'].shape, b['y'].shape, b['m'].sum().item())

## 7. Model: Age-Aware Gated-Fusion Hybrid (AGF-Net)
For an image $x$ with backbones $k=1..K$ (EfficientNet-B0, MobileViTv2-1.0; about 9.5 M parameters in total):

* projected features $f_k=\mathrm{GELU}(\mathrm{LN}(W_k\,\mathrm{pool}(B_k(x))))$, sample-wise gate $w=\mathrm{softmax}(G[f_1;\dots;f_K])$, fused $z=\sum_k w_k f_k$;
* age-group head $\hat a=\sigma(h_a^\top z)$; age signal $a$ = known age group (hidden with prob. `AGE_DROP` during training) or $\hat a$;
* FiLM: $z' = z\odot(1+\gamma(a))+\beta(a)$ (last layer initialised to zero → starts as identity);
* disease logits $s(x)=W_c z' + \tau\,[(1-a)\,\mathrm{logit}\,\pi_{adult}+a\,\mathrm{logit}\,\pi_{ped}]$ (age-specific prevalence offset);
* pediatric pneumonia subtype head (bacterial / viral);
* **hospital-adversarial head** (`joint_age_adv` only): a small discriminator predicts, for children, whether the image came from NIH or Guangzhou; it sees $z$ through a **gradient-reversal layer**, so training pushes $z$ to *hide* the hospital. Its strength ramps from 0 to 1 during training (DANN schedule). It is used only during training and is not part of the deployed model.

Loss: masked, class-weighted BCE + $\lambda_{age}$ BCE(age) + $\lambda_{sub}$ CE(subtype) + $\lambda_{adv}$ BCE(hospital, through gradient reversal).

In [ ]:
class Branch(nn.Module):
    def __init__(self, name, pretrained):
        super().__init__()
        kw = {'img_size': ISZ} if (ISZ != 224 and any(k in name for k in ('swin', 'vit', 'eva'))) else {}
        self.net = timm.create_model(name, pretrained=pretrained, num_classes=0, **kw)
        self.nf = self.net.num_features
        if CFG['GRAD_CHECKPOINT']:
            try: self.net.set_grad_checkpointing(True)
            except Exception: pass
    def forward(self, x):
        f = self.net.forward_features(x)
        return f, self.net.forward_head(f, pre_logits=True)

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, coef):
        ctx.coef = coef; return x.view_as(x)
    @staticmethod
    def backward(ctx, g):
        return -ctx.coef * g, None

class AGFNet(nn.Module):
    def __init__(self, names, n_labels, proj=512, dropout=0.3, film=True, age_head=True, sub_head=True,
                 logit_adj=True, prior_logit=None, pretrained=True, adv=False):
        super().__init__()
        self.branches = nn.ModuleList([Branch(n, pretrained) for n in names])
        self.projs = nn.ModuleList([nn.Sequential(nn.Linear(b.nf, proj), nn.LayerNorm(proj), nn.GELU()) for b in self.branches])
        K = len(names)
        self.gate = nn.Sequential(nn.Linear(proj * K, 256), nn.GELU(), nn.Linear(256, K)) if K > 1 else None
        self.age_head = nn.Linear(proj, 1) if age_head else None
        self.film = nn.Sequential(nn.Linear(1, 64), nn.GELU(), nn.Linear(64, 2 * proj)) if film else None
        if film:
            nn.init.zeros_(self.film[-1].weight); nn.init.zeros_(self.film[-1].bias)
        self.head = nn.Sequential(nn.LayerNorm(proj), nn.Dropout(dropout), nn.Linear(proj, n_labels))
        self.sub_head = nn.Linear(proj, 2) if sub_head else None
        # hospital discriminator: tries to tell NIH children from Guangzhou children; gradient reversal makes z hide the hospital
        self.src_head = nn.Sequential(nn.Linear(proj, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 1)) if adv else None
        self.logit_adj = logit_adj
        self.register_buffer('prior_logit', torch.tensor(prior_logit if prior_logit is not None else np.zeros((2, n_labels)), dtype=torch.float32))
        self.tau = CFG['LOGIT_ADJ_TAU']

    def backbone_params(self):
        return [p for b in self.branches for p in b.parameters()]

    def forward(self, x, age=None, age_known=None, age_drop=0.0, return_maps=False, grl=0.0):
        maps, feats = [], []
        for b, pj in zip(self.branches, self.projs):
            f, p = b(x); maps.append(f); feats.append(pj(p))
        Fs = torch.stack(feats, 1)
        w = torch.softmax(self.gate(Fs.flatten(1)), -1) if self.gate is not None else torch.ones(Fs.shape[:2], device=x.device)
        z = (w.unsqueeze(-1) * Fs).sum(1)
        age_logit = self.age_head(z).squeeze(-1) if self.age_head is not None else None
        a = None
        if self.film is not None or self.logit_adj:
            a = torch.sigmoid(age_logit).detach() if age_logit is not None else torch.full((x.shape[0],), 0.5, device=x.device)
            if age is not None:
                known = age_known > 0.5
                if self.training and age_drop > 0:
                    known = known & (torch.rand_like(age) > age_drop)
                a = torch.where(known, age.float(), a)
        zc = z
        if self.film is not None:
            g, bb = self.film(a.unsqueeze(-1)).chunk(2, -1)
            zc = z * (1 + g) + bb
        logits = self.head(zc)
        if self.logit_adj:
            logits = logits + self.tau * ((1 - a)[:, None] * self.prior_logit[0] + a[:, None] * self.prior_logit[1])
        out = dict(logits=logits, age_logit=age_logit, gate=w, z=z,
                   sub_logits=self.sub_head(zc) if self.sub_head is not None else None,
                   src_logit=self.src_head(GradReverse.apply(z, grl)).squeeze(-1) if (self.src_head is not None and self.training) else None)
        if return_maps:
            out['maps'] = maps
        return out

def build_model(pretrained=None):
    return AGFNet(CFG['BACKBONES'], NL, CFG['PROJ_DIM'], CFG['DROPOUT'], film=FLAGS['film'], age_head=FLAGS['age_head'],
                  sub_head=FLAGS['sub_head'], logit_adj=FLAGS['logit_adj'], prior_logit=PRIOR_LOGIT,
                  pretrained=CFG['PRETRAINED'] if pretrained is None else pretrained, adv=FLAGS['adv'])

model = build_model().to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f} M')
with torch.no_grad():
    o = model(b['x'][:2].to(DEVICE), b['age'][:2].to(DEVICE), b['age_known'][:2].to(DEVICE))
print({k: tuple(v.shape) for k, v in o.items() if v is not None})

# How lightweight is the model?
n_par = sum(p.numel() for p in model.parameters())
model.eval(); xb_ = torch.randn(1, 3, ISZ, ISZ, device=DEVICE)
with torch.no_grad():
    for _ in range(3): model(xb_)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    t_ = time.time()
    for _ in range(20): model(xb_)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
LIGHT = dict(parameters_M=round(n_par / 1e6, 2), size_fp32_MB=round(n_par * 4 / 1e6, 1), size_fp16_MB=round(n_par * 2 / 1e6, 1),
             ms_per_image=round((time.time() - t_) / 20 * 1000, 1), device=str(DEVICE))
print('Lightweight check:', LIGHT, '| one model serves adults and children')

## 8. Training (two-stage: frozen backbones, then end-to-end) with EMA, AMP, early stopping and resume
Model selection uses only the validation split: **adult val macro-AUROC** and **pediatric val macro-AUROC**, averaged for the joint experiments (each single-domain experiment uses its own domain). The test splits are never touched here.

In [ ]:
def masked_bce(logits, y, m, pw):
    l = F.binary_cross_entropy_with_logits(logits, y, reduction='none', pos_weight=pw)
    return (l * m).sum() / m.sum().clamp(min=1)

def compute_loss(out, bt):
    loss = masked_bce(out['logits'].float(), bt['y'], bt['m'], POS_WEIGHT.to(DEVICE))
    if out['age_logit'] is not None:
        k = bt['age_known'] > 0.5
        if k.any():
            loss = loss + CFG['LAMBDA_AGE'] * F.binary_cross_entropy_with_logits(out['age_logit'][k].float(), bt['age'][k])
    if out['sub_logits'] is not None and (bt['sub'] >= 0).any():
        loss = loss + CFG['LAMBDA_SUB'] * F.cross_entropy(out['sub_logits'].float(), bt['sub'], ignore_index=-1)
    if out.get('src_logit') is not None:
        kp = (bt['age_known'] > 0.5) & (bt['age'] > 0.5)          # children only: adults all come from NIH
        if kp.any() and 0 < bt['src'][kp].sum() < kp.sum():
            loss = loss + CFG['LAMBDA_ADV'] * F.binary_cross_entropy_with_logits(out['src_logit'][kp].float(), bt['src'][kp])
    return loss

def macro_auc(y, p, m, min_pos=1):
    aucs = []
    for c in range(y.shape[1]):
        k = m[:, c] > 0
        yc = y[k, c]
        if yc.sum() >= min_pos and (1 - yc).sum() >= min_pos:
            aucs.append(roc_auc_score(yc, p[k, c]))
    return float(np.mean(aucs)) if aucs else float('nan')

@torch.no_grad()
def predict(net, idx, corruption=None, use_age=True, maps=False):
    net.eval(); L, A, Sb, G, Z = [], [], [], [], []
    for bt in make_loader(idx, corruption):
        x = bt['x'].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            o = net(x, bt['age'].to(DEVICE) if use_age else None, bt['age_known'].to(DEVICE) if use_age else None)
        L.append(o['logits'].float().cpu()); G.append(o['gate'].float().cpu()); Z.append(o['z'].float().cpu())
        A.append(o['age_logit'].float().cpu() if o['age_logit'] is not None else torch.zeros(len(x)))
        Sb.append(o['sub_logits'].float().cpu() if o['sub_logits'] is not None else torch.zeros(len(x), 2))
    cat = lambda t: torch.cat(t).numpy()
    return dict(logits=cat(L), age_logit=cat(A), sub_logits=cat(Sb), gate=cat(G), z=cat(Z))

def val_score(pr, idx):
    p = 1 / (1 + np.exp(-pr['logits'])); y, m = Yraw[idx], (~np.isnan(Yraw[idx])).astype(float); y = np.nan_to_num(y)
    g = AGEG[idx]
    s_ad = macro_auc(y[g == 0], p[g == 0], m[g == 0]) if (g == 0).any() else float('nan')
    s_pe = macro_auc(y[g == 1], p[g == 1], m[g == 1]) if (g == 1).any() else float('nan')
    sel = np.nanmean([s_ad if FLAGS['use_adult'] else np.nan, s_pe if FLAGS['use_ped'] else np.nan])
    return sel, s_ad, s_pe

class EMA:
    def __init__(self, net, decay):
        self.m = copy.deepcopy(net).eval(); self.d = decay
        for p in self.m.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, net):
        for e, p in zip(self.m.state_dict().values(), net.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(self.d).add_(p.detach(), alpha=1 - self.d)
            else: e.copy_(p)

def make_scaler():
    try: return torch.amp.GradScaler('cuda', enabled=AMP)
    except Exception: return torch.cuda.amp.GradScaler(enabled=AMP)

def train():
    net = model
    bb_ids = {id(p) for p in net.backbone_params()}
    head_params = [p for p in net.parameters() if id(p) not in bb_ids]
    opt = torch.optim.AdamW([{'params': net.backbone_params(), 'lr': CFG['LR_BACKBONE']},
                             {'params': head_params, 'lr': CFG['LR_HEAD']}], weight_decay=CFG['WD'])
    loader = make_train_loader(); spe = len(loader); total = CFG['EPOCHS'] * spe; warm = max(1, int(total * CFG['WARMUP_FRAC']))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1, s / total))))
    scaler, ema = make_scaler(), EMA(net, CFG['EMA_DECAY'])
    hist, best, start, bad = [], -1, 0, 0
    if (OUT / 'last.pt').exists():
        ck = torch.load(OUT / 'last.pt', map_location=DEVICE, weights_only=False)
        net.load_state_dict(ck['model']); ema.m.load_state_dict(ck['ema']); opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
        hist, best, start, bad = ck['hist'], ck['best'], ck['epoch'] + 1, ck['bad']
        print(f'Resumed from epoch {start}')
    t_start = time.time()
    for ep in range(start, CFG['EPOCHS']):
        frozen = ep < CFG['FREEZE_EPOCHS']
        for p in net.backbone_params(): p.requires_grad_(not frozen)
        net.train(); t0 = time.time(); run = 0.0; corr = 0.0; cnt = 0.0
        for it, bt in enumerate(loader):
            bt = {k: v.to(DEVICE, non_blocking=True) for k, v in bt.items()}
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                prog = (ep * spe + it) / max(total, 1)
                out = net(bt['x'], bt['age'], bt['age_known'], age_drop=CFG['AGE_DROP'],
                          grl=2.0 / (1.0 + math.exp(-10 * prog)) - 1.0)      # DANN ramp 0 -> 1
            loss = compute_loss(out, bt)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), CFG['GRAD_CLIP'])
            scaler.step(opt); scaler.update(); sched.step(); ema.update(net)
            run += loss.item()
            with torch.no_grad():
                mk_ = bt['m'] > 0
                corr += (((out['logits'].float() > 0) == (bt['y'] > 0.5)) & mk_).sum().item(); cnt += mk_.sum().item()
            if (it + 1) % 200 == 0:
                print(f'  ep{ep} it{it+1}/{spe} loss {run/(it+1):.4f} {time.time()-t0:.0f}s')
        pr = predict(ema.m, va_idx)
        sel, s_ad, s_pe = val_score(pr, va_idx)
        yv_, mv_ = np.nan_to_num(Yraw[va_idx]), (~np.isnan(Yraw[va_idx])).astype(np.float32)
        v_loss = float(masked_bce(torch.tensor(pr['logits']), torch.tensor(yv_), torch.tensor(mv_), POS_WEIGHT))
        pv_, mb_ = pr['logits'] > 0, mv_ > 0
        tp_, tn_ = (pv_ & (yv_ == 1) & mb_).sum(), (~pv_ & (yv_ == 0) & mb_).sum()
        v_acc = (tp_ + tn_) / max(mb_.sum(), 1)
        v_sen, v_spe = tp_ / max(((yv_ == 1) & mb_).sum(), 1), tn_ / max(((yv_ == 0) & mb_).sum(), 1)
        hist.append(dict(epoch=ep, loss=run / spe, train_loss=run / spe, train_acc=corr / max(cnt, 1), val_loss=v_loss, val_acc=float(v_acc),
                         val_sensitivity=float(v_sen), val_bal_acc=float((v_sen + v_spe) / 2), val_sel=sel, val_adult_auc=s_ad,
                         val_ped_auc=s_pe, frozen=frozen, sec=time.time() - t0))
        print(f'Epoch {ep}: train loss {run/spe:.4f} acc {corr/max(cnt,1):.4f} | val loss {v_loss:.4f} acc {v_acc:.4f} '
              f'bal-acc {(v_sen+v_spe)/2:.4f} | val mAUROC adult {s_ad:.4f} ped {s_pe:.4f} | {time.time()-t0:.0f}s')
        if sel > best:
            best, bad = sel, 0; torch.save(ema.m.state_dict(), OUT / 'best.pt')
        else:
            bad += 1
        torch.save(dict(model=net.state_dict(), ema=ema.m.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), hist=hist, best=best, epoch=ep, bad=bad), OUT / 'last.pt')
        if bad >= CFG['PATIENCE'] or (time.time() - t_start) / 3600 > CFG['MAX_TRAIN_HOURS']:
            print('Stopping (patience or time budget).'); break
    return pd.DataFrame(hist)

hist = train()
hist.to_csv(OUT / 'history.csv', index=False)
e_ = hist.epoch + 1
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].plot(e_, hist.train_loss, 'o-', label='train'); ax[0].plot(e_, hist.val_loss, 's-', label='validation')
ax[0].set_title('Loss vs epoch'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('masked BCE loss'); ax[0].legend()
ax[1].plot(e_, hist.train_acc, 'o-', label='train accuracy'); ax[1].plot(e_, hist.val_acc, 's-', label='validation accuracy')
ax[1].plot(e_, hist.val_bal_acc, '^--', label='validation balanced accuracy')
ax[1].set_title('Accuracy vs epoch (threshold 0.5)'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy'); ax[1].legend()
ax[2].plot(e_, hist.val_adult_auc, 'o-', label='adult'); ax[2].plot(e_, hist.val_ped_auc, 's-', label='pediatric')
ax[2].set_title('Validation macro-AUROC'); ax[2].set_xlabel('epoch'); ax[2].legend()
for a_ in ax: a_.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG / 'training_curves_loss_accuracy.png', dpi=130); plt.show()

## 9. Calibration and age-specific thresholds (validation only)
Per disease and per **age band** (adult 18+, child 6-17, child 0-5, child of unknown age): a temperature minimising NLL, then a threshold maximising Youden's J. Each disease is calibrated **only on the datasets that contain it** (Guangzhou images are used for Pneumonia only). Bands with fewer than 10 positives or negatives fall back: child band → all children → adult → pooled. An **image-level abnormal cut-off** is also fitted, so "abnormal" is no longer "any of 14 thresholds fired". All numbers are frozen **before** the test set is opened.

In [ ]:
model.load_state_dict(torch.load(OUT / 'best.pt', map_location=DEVICE))
model.eval()
pr_val = predict(model, va_idx, use_age=CFG['INFER_USE_KNOWN_AGE'])

def fit_temperature(z, y):
    t = torch.zeros(1, requires_grad=True); zt, yt = torch.tensor(z), torch.tensor(y)
    opt = torch.optim.LBFGS([t], lr=0.1, max_iter=100)
    def clo():
        opt.zero_grad(); l = F.binary_cross_entropy_with_logits(zt / t.exp(), yt); l.backward(); return l
    opt.step(clo)
    return float(np.clip(t.exp().item(), 0.3, 5.0))

def youden_thr(y, p):
    fpr, tpr, th = roc_curve(y, p); k = np.argmax(tpr - fpr); return float(np.clip(th[k], 1e-4, 1 - 1e-4))

# Calibration groups: 0 = adult (18+), 1 = child 6-17, 2 = child 0-5, 3 = child of unknown age (all children pooled)
GNAMES = ['adult', 'child 6-17', 'child 0-5', 'child (age unknown)']
CALG = np.where(meta.source.values == 'KER', 2, np.where(meta.age.values < 6, 2, np.where(meta.age.values < 18, 1,
                np.where(meta.age.values >= 18, 0, -1)))).astype(int)

def group_of_age(age, ped_prob=None):
    if age is not None and age >= 0:
        return 2 if age < 6 else (1 if age < 18 else 0)
    return 3 if (ped_prob is not None and ped_prob > 0.5) else 0

yv, mv = np.nan_to_num(Yraw[va_idx]), ~np.isnan(Yraw[va_idx])
gv, src_v = CALG[va_idx], meta.source.values[va_idx]
TEMP = np.ones((4, NL), np.float32); THR = np.full((4, NL), 0.5, np.float32)

def fit_pair(k, c):
    # temperature + Youden threshold if at least 10 positives and 10 negatives, else None
    if k.sum() == 0 or not (10 <= yv[k, c].sum() <= k.sum() - 10):
        return None
    t_ = fit_temperature(pr_val['logits'][k, c], yv[k, c])
    return t_, youden_thr(yv[k, c], 1 / (1 + np.exp(-pr_val['logits'][k, c] / t_)))

calib_log = []
for c in range(NL):
    # Rule: calibrate each disease only on the datasets that contain that disease in validation.
    # (Guangzhou images never contain Infiltration etc.; using them made pediatric thresholds far too low for NIH children.)
    srcs = [sv for sv in np.unique(src_v) if (mv[:, c] & (src_v == sv) & (yv[:, c] == 1)).any()]
    k_all = mv[:, c] & np.isin(src_v, srcs)
    pool = fit_pair(k_all, c) or (1.0, 0.5)
    adult = fit_pair(k_all & (gv == 0), c) or pool
    child = fit_pair(k_all & np.isin(gv, [1, 2]), c) or adult
    g1 = fit_pair(k_all & (gv == 1), c) or child
    g2 = fit_pair(k_all & (gv == 2), c) or child
    for g, (t_, th_) in enumerate([adult, g1, g2, child]):
        TEMP[g, c], THR[g, c] = t_, th_
    calib_log.append(dict(label=LABELS[c], sources=','.join(srcs)))
CAL_TAB = pd.DataFrame({f'T {n}': TEMP[g] for g, n in enumerate(GNAMES)} | {f'thr {n}': THR[g] for g, n in enumerate(GNAMES)}, index=LABELS)
CAL_TAB['calibrated on'] = [d['sources'] for d in calib_log]
print(CAL_TAB.round(3).to_string())

def calibrate(logits, groups):
    g = np.where(np.asarray(groups) < 0, 0, np.asarray(groups)).astype(int)
    return 1 / (1 + np.exp(-logits / TEMP[g])), THR[g]

# Image-level "abnormal" decision: one score per image = the largest margin above its own threshold
# (logit of calibrated probability minus logit of the threshold); its cut-off is fitted on validation (Youden).
def lgt(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def abn_score(P, T): return (lgt(P) - lgt(T)).max(1)
Pv_, Tv_ = calibrate(pr_val['logits'], gv)
sv_ = abn_score(Pv_, Tv_); yabn_v = ((yv == 1) & mv).any(1)
ABN_THR = np.zeros(2, np.float32)          # 0 = adult, 1 = child
for gi, kk in enumerate([gv == 0, np.isin(gv, [1, 2, 3])]):
    if kk.sum() and 0 < yabn_v[kk].sum() < kk.sum():
        f_, t_, th_ = roc_curve(yabn_v[kk], sv_[kk])
        j_ = 1 + np.argmax((t_ - f_)[1:])        # skip roc_curve's first (infinite) threshold
        ABN_THR[gi] = th_[j_]
ABN_THR = np.clip(ABN_THR, -20, 20)
print('Image-level abnormal cut-off (margin): adult %.3f | child %.3f' % tuple(ABN_THR))
np.savez(OUT / 'calibration.npz', TEMP=TEMP, THR=THR, ABN_THR=ABN_THR, PRIOR_LOGIT=PRIOR_LOGIT, LABELS=np.array(LABELS))

def abnormal(P, T, groups):
    g = np.asarray(groups); return abn_score(P, T) >= ABN_THR[np.where(np.isin(g, [1, 2, 3]), 1, 0)]

## 10. Locked test evaluation (run once)
Test sets: **NIH-adult**, **NIH-pediatric** (children inside NIH, same hospital), **Pediatric-GWCMC** (Kermany test), and the optional **External** set. Metrics: per-label and macro AUROC, AUPRC, sensitivity, specificity, F1 at the frozen thresholds, ECE before/after temperature scaling, and 95% CIs from a **patient-cluster bootstrap**.

In [ ]:
pr_te = predict(model, te_idx, use_age=CFG['INFER_USE_KNOWN_AGE'])
pr_te_noage = predict(model, te_idx, use_age=False) if (FLAGS['film'] or FLAGS['logit_adj']) else pr_te
P_raw = 1 / (1 + np.exp(-pr_te['logits']))
P_cal, T_te = calibrate(pr_te['logits'], CALG[te_idx])
ABN_te = abnormal(P_cal, T_te, CALG[te_idx])           # image-level normal / abnormal decision
np.savez_compressed(OUT / 'test_predictions.npz', uid=meta.uid.values[te_idx], logits=pr_te['logits'], logits_noage=pr_te_noage['logits'],
                    P_cal=P_cal, thr=T_te, age_logit=pr_te['age_logit'], sub_logits=pr_te['sub_logits'], gate=pr_te['gate'])

def ece_binary(y, p, bins=15):
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for a, b_ in zip(edges[:-1], edges[1:]):
        k = (p > a) & (p <= b_)
        if k.any(): e += k.mean() * abs(y[k].mean() - p[k].mean())
    return e

def evaluate(rows, P, Pr, T):
    y, m = np.nan_to_num(Yraw[rows]), ~np.isnan(Yraw[rows])
    per = []
    for c, L in enumerate(LABELS):
        k = m[:, c]; yc = y[k, c]
        if k.sum() == 0 or yc.sum() == 0 or yc.sum() == k.sum():
            continue
        pc, pred = P[k, c], P[k, c] >= T[k, c]
        tp, fn, fp, tn = (pred & (yc == 1)).sum(), (~pred & (yc == 1)).sum(), (pred & (yc == 0)).sum(), (~pred & (yc == 0)).sum()
        per.append(dict(label=L, n=int(k.sum()), pos=int(yc.sum()), AUROC=roc_auc_score(yc, pc), AUPRC=average_precision_score(yc, pc),
                        sens=tp / max(tp + fn, 1), spec=tn / max(tn + fp, 1), F1=2 * tp / max(2 * tp + fp + fn, 1),
                        ECE_raw=ece_binary(yc, Pr[k, c]), ECE_cal=ece_binary(yc, pc)))
    return pd.DataFrame(per)

def cluster_bootstrap_auc(rows, P, n_boot, seed=0):
    pids = meta.patient_id.values[rows]; up, inv = np.unique(pids, return_inverse=True)
    members = [np.where(inv == i)[0] for i in range(len(up))]
    rng, y, m, vals = np.random.default_rng(seed), np.nan_to_num(Yraw[rows]), (~np.isnan(Yraw[rows])).astype(float), []
    for _ in range(n_boot):
        s = np.concatenate([members[i] for i in rng.integers(0, len(up), len(up))])
        vals.append(macro_auc(y[s], P[s], m[s]))
    return np.nanpercentile(vals, [2.5, 97.5])

TEST_SETS = [t for t in ['NIH-adult', 'NIH-pediatric', 'Pediatric-GWCMC', 'External'] if (meta.test_set.values[te_idx] == t).any()]
summary, per_label = {}, []
for ts in TEST_SETS:
    k = np.where(meta.test_set.values[te_idx] == ts)[0]; rows = te_idx[k]
    df = evaluate(rows, P_cal[k], P_raw[k], T_te[k]); df.insert(0, 'test_set', ts); per_label.append(df)
    y, mm = np.nan_to_num(Yraw[rows]), (~np.isnan(Yraw[rows])).astype(float)
    ci = cluster_bootstrap_auc(rows, P_cal[k], CFG['N_BOOT'])
    summary[ts] = dict(n_images=len(rows), n_patients=len(set(meta.patient_id.values[rows])),
                       macro_AUROC=macro_auc(y, P_cal[k], mm), CI95=list(map(float, ci)),
                       macro_AUROC_without_age=macro_auc(y, 1 / (1 + np.exp(-pr_te_noage['logits'][k])), mm),
                       macro_AUPRC=float(df.AUPRC.mean()) if len(df) else float('nan'),
                       macro_sens=float(df.sens.mean()) if len(df) else float('nan'), macro_spec=float(df.spec.mean()) if len(df) else float('nan'),
                       macro_F1=float(df.F1.mean()) if len(df) else float('nan'),
                       ECE_raw=float(df.ECE_raw.mean()) if len(df) else float('nan'), ECE_cal=float(df.ECE_cal.mean()) if len(df) else float('nan'),
                       pneumonia_AUROC=float(df.set_index('label').AUROC.get('Pneumonia', np.nan)) if len(df) else float('nan'))
per_label = pd.concat(per_label, ignore_index=True)
per_label.to_csv(OUT / 'test_per_label.csv', index=False)
S_df = pd.DataFrame(summary).T
print(S_df.round(4).to_string()); print()
print(per_label.round(3).to_string())

In [ ]:
# Pediatric GWCMC: normal vs pneumonia confusion matrix and bacterial/viral subtype accuracy
if 'Pediatric-GWCMC' in TEST_SETS:
    k = np.where(meta.test_set.values[te_idx] == 'Pediatric-GWCMC')[0]; rows = te_idx[k]
    yt, pp, th = Yraw[rows, PNEU].astype(int), P_cal[k, PNEU], T_te[k, PNEU]
    cm = confusion_matrix(yt, pp >= th)
    acc = (cm[0, 0] + cm[1, 1]) / cm.sum()
    summary['Pediatric-GWCMC'].update(accuracy=float(acc), sensitivity=float(cm[1, 1] / cm[1].sum()), specificity=float(cm[0, 0] / cm[0].sum()))
    st = meta.subtype.values[rows]; ks = st >= 0
    if FLAGS['sub_head'] and ks.any():
        summary['Pediatric-GWCMC']['subtype_accuracy'] = float((pr_te['sub_logits'][k][ks].argmax(1) == st[ks]).mean())
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm): ax[0].text(j, i, v, ha='center', va='center', fontsize=14)
    ax[0].set_xticks([0, 1], ['Normal', 'Pneumonia']); ax[0].set_yticks([0, 1], ['Normal', 'Pneumonia'])
    ax[0].set_xlabel('predicted'); ax[0].set_ylabel('true'); ax[0].set_title(f'Pediatric test: acc {acc:.3f}')
    for ts in TEST_SETS:
        kk = np.where(meta.test_set.values[te_idx] == ts)[0]; yy = Yraw[te_idx[kk], PNEU]; ok = ~np.isnan(yy)
        if ok.any() and 0 < yy[ok].sum() < ok.sum():
            f_, t_, _ = roc_curve(yy[ok], P_cal[kk][ok, PNEU]); ax[1].plot(f_, t_, label=f'{ts} AUC {roc_auc_score(yy[ok], P_cal[kk][ok, PNEU]):.3f}')
    ax[1].plot([0, 1], [0, 1], 'k:'); ax[1].legend(fontsize=8); ax[1].set_title('Pneumonia ROC by test set')
    plt.tight_layout(); plt.savefig(FIG / 'pediatric_cm_pneumonia_roc.png', dpi=120); plt.show()
    print({k_: v for k_, v in summary['Pediatric-GWCMC'].items() if k_ in ('accuracy', 'sensitivity', 'specificity', 'subtype_accuracy')})

# per-label AUROC heat map across test sets
pv = per_label.pivot(index='label', columns='test_set', values='AUROC').reindex(LABELS)
plt.figure(figsize=(6, 6)); plt.imshow(pv.values, cmap='viridis', vmin=0.5, vmax=1, aspect='auto')
plt.colorbar(label='AUROC'); plt.yticks(range(NL), LABELS); plt.xticks(range(pv.shape[1]), pv.columns, rotation=30)
for (i, j), v in np.ndenumerate(pv.values):
    if not np.isnan(v): plt.text(j, i, f'{v:.2f}', ha='center', va='center', color='w', fontsize=8)
plt.title(f'Per-label AUROC ({EXP})'); plt.tight_layout(); plt.savefig(FIG / 'per_label_auroc.png', dpi=120); plt.show()

### 10.1 Overall accuracy, sensitivity, specificity, precision, F1 (one number per test set)
* **Overall (micro):** every disease decision on every image counted together.
* **Normal vs abnormal:** each image counted once: does it have any disease or not?
* **Per disease:** accuracy, precision, recall (= sensitivity), specificity and F1 for each of the 14 diseases.

All use the age-specific thresholds fixed on validation data. In multi-label data accuracy looks high because most diseases are absent in most images, so always report sensitivity next to it.

In [ ]:
def overall(rows_k):
    r = te_idx[rows_k]; y, P, T = Yraw[r], P_cal[rows_k], T_te[rows_k]
    m = ~np.isnan(y); yt = np.nan_to_num(y).astype(int); yp = (P >= T).astype(int)
    tp = ((yp == 1) & (yt == 1) & m).sum(); tn = ((yp == 0) & (yt == 0) & m).sum()
    fp = ((yp == 1) & (yt == 0) & m).sum(); fn = ((yp == 0) & (yt == 1) & m).sum()
    true_abn = ((yt == 1) & m).any(1); pred_abn = ABN_te[rows_k]
    return {'images': len(r),
            'Accuracy (overall)': (tp + tn) / max(tp + tn + fp + fn, 1),
            'Sensitivity (overall)': tp / max(tp + fn, 1),
            'Specificity (overall)': tn / max(tn + fp, 1),
            'Precision (overall)': tp / max(tp + fp, 1),
            'F1 (overall)': 2 * tp / max(2 * tp + fp + fn, 1),
            'Normal-vs-abnormal accuracy': float((true_abn == pred_abn).mean()),
            'Normal-vs-abnormal sensitivity': (pred_abn & true_abn).sum() / max(true_abn.sum(), 1),
            'Normal-vs-abnormal specificity': (~pred_abn & ~true_abn).sum() / max((~true_abn).sum(), 1)}

OV = {ts: overall(np.where(meta.test_set.values[te_idx] == ts)[0]) for ts in TEST_SETS}
OV['ALL TEST IMAGES'] = overall(np.arange(len(te_idx)))
OV = pd.DataFrame(OV).T
OV.to_csv(OUT / 'overall_accuracy_sensitivity.csv')
for ts in TEST_SETS:
    summary[ts].update({k: float(v) for k, v in OV.loc[ts].items() if k != 'images'})
print('OVERALL RESULTS PER TEST SET')
print(OV.round(4).to_string())

rows = []
for ts in TEST_SETS:
    k = np.where(meta.test_set.values[te_idx] == ts)[0]; r = te_idx[k]
    y, P, T = Yraw[r], P_cal[k], T_te[k]
    for c, L in enumerate(LABELS):
        m = ~np.isnan(y[:, c])
        if m.sum() == 0 or y[m, c].sum() == 0:
            continue
        yt = y[m, c].astype(int); yp = (P[m, c] >= T[m, c]).astype(int)
        tp = ((yp == 1) & (yt == 1)).sum(); fp = ((yp == 1) & (yt == 0)).sum()
        fn = ((yp == 0) & (yt == 1)).sum(); tn = ((yp == 0) & (yt == 0)).sum()
        rows.append(dict(test_set=ts, label=L, n=len(yt), positives=int(yt.sum()), accuracy=(tp + tn) / len(yt),
                         precision=tp / max(tp + fp, 1), recall_sensitivity=tp / max(tp + fn, 1),
                         specificity=tn / max(tn + fp, 1), F1=2 * tp / max(2 * tp + fp + fn, 1)))
M = pd.DataFrame(rows); M.to_csv(OUT / 'test_accuracy_precision_recall.csv', index=False)
print('\nPER DISEASE')
print(M.round(3).to_string())
print('\nAVERAGE OVER DISEASES (macro)')
print(M.groupby('test_set')[['accuracy', 'precision', 'recall_sensitivity', 'specificity', 'F1']].mean().round(3))

### 10.2 Confusion matrices
For each test set: (a) **normal vs abnormal** (any of the 14 diseases), (b) a 2×2 confusion matrix for **every disease** that has positive cases, at the age-specific thresholds. For the pediatric Guangzhou set the normal-vs-pneumonia matrix is in the cell above, and the bacterial-vs-viral matrix is drawn here.

In [ ]:
def draw_cm(ax, cm_, labels_, title_, fs=11):
    ax.imshow(cm_, cmap='Blues')
    tot = cm_.sum()
    for (i, j), v in np.ndenumerate(cm_):
        ax.text(j, i, f'{v}\n{v / max(tot, 1):.1%}', ha='center', va='center', fontsize=fs,
                color='white' if v > cm_.max() * 0.6 else 'black')
    ax.set_xticks([0, 1], labels_); ax.set_yticks([0, 1], labels_)
    ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title(title_, fontsize=fs)

for ts in TEST_SETS:
    k = np.where(meta.test_set.values[te_idx] == ts)[0]; r = te_idx[k]
    y, P, T = Yraw[r], P_cal[k], T_te[k]; m = ~np.isnan(y); yt = np.nan_to_num(y).astype(int); yp = (P >= T).astype(int)
    true_abn, pred_abn = ((yt == 1) & m).any(1), ABN_te[k]
    cm_na = confusion_matrix(true_abn, pred_abn, labels=[False, True])
    labs = [c for c in range(NL) if m[:, c].any() and yt[m[:, c], c].sum() > 0]
    ncol = 5; nrow = 1 + int(np.ceil(len(labs) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.1 * nrow)); axs = np.atleast_2d(axs)
    for a_ in axs.ravel(): a_.axis('off')
    axs[0, 0].axis('on'); draw_cm(axs[0, 0], cm_na, ['Normal', 'Abnormal'], f'{ts}\nnormal vs abnormal')
    for i, c in enumerate(labs):
        a_ = axs[1 + i // ncol, i % ncol]; a_.axis('on'); mc = m[:, c]
        draw_cm(a_, confusion_matrix(yt[mc, c], yp[mc, c], labels=[0, 1]), ['neg', 'pos'], LABELS[c], fs=9)
    plt.suptitle(f'Confusion matrices: {ts}', fontsize=13); plt.tight_layout()
    plt.savefig(FIG / f'confusion_matrices_{ts}.png', dpi=120); plt.show()

# bacterial vs viral (children with pneumonia in the Guangzhou test set)
if FLAGS['sub_head'] and 'Pediatric-GWCMC' in TEST_SETS:
    k = np.where(meta.test_set.values[te_idx] == 'Pediatric-GWCMC')[0]; st = meta.subtype.values[te_idx[k]]; ks = st >= 0
    if ks.any():
        cm_sub = confusion_matrix(st[ks], pr_te['sub_logits'][k][ks].argmax(1), labels=[0, 1])
        fig, ax = plt.subplots(figsize=(4, 3.6)); draw_cm(ax, cm_sub, ['Bacterial', 'Viral'], 'Pediatric pneumonia subtype')
        plt.tight_layout(); plt.savefig(FIG / 'confusion_matrix_subtype.png', dpi=120); plt.show()

### 10.3 ROC curves
For each test set: one ROC curve per disease (with its AUROC) plus the **micro-average** over all disease decisions, and the matching precision-recall curves.

In [ ]:
cmap_ = plt.get_cmap('tab20')
for ts in TEST_SETS:
    k = np.where(meta.test_set.values[te_idx] == ts)[0]; r = te_idx[k]
    y, P = Yraw[r], P_cal[k]; m = ~np.isnan(y); yt = np.nan_to_num(y).astype(int)
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    for c in range(NL):
        mc = m[:, c]
        if mc.sum() == 0 or yt[mc, c].sum() == 0 or yt[mc, c].sum() == mc.sum(): continue
        f_, t_, _ = roc_curve(yt[mc, c], P[mc, c]); au = roc_auc_score(yt[mc, c], P[mc, c])
        ax[0].plot(f_, t_, color=cmap_(c), lw=1.3, label=f'{LABELS[c]} ({au:.3f})')
        from sklearn.metrics import precision_recall_curve
        pr_, rc_, _ = precision_recall_curve(yt[mc, c], P[mc, c])
        ax[1].plot(rc_, pr_, color=cmap_(c), lw=1.3, label=f'{LABELS[c]} ({average_precision_score(yt[mc, c], P[mc, c]):.3f})')
    yy, pp = yt[m], P[m]
    if 0 < yy.sum() < len(yy):
        f_, t_, _ = roc_curve(yy, pp)
        ax[0].plot(f_, t_, 'k--', lw=2.2, label=f'micro-average ({roc_auc_score(yy, pp):.3f})')
    ax[0].plot([0, 1], [0, 1], ':', color='grey')
    ax[0].set_xlabel('False positive rate (1 - specificity)'); ax[0].set_ylabel('True positive rate (sensitivity)')
    ax[0].set_title(f'ROC curves: {ts}'); ax[0].legend(fontsize=8, loc='lower right')
    ax[1].set_xlabel('Recall (sensitivity)'); ax[1].set_ylabel('Precision'); ax[1].set_title(f'Precision-recall curves: {ts}')
    ax[1].legend(fontsize=8, loc='upper right')
    for a_ in ax: a_.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(FIG / f'roc_pr_curves_{ts}.png', dpi=120); plt.show()

### 10.4 Results at a glance
One table with every headline number per test set. The same single model produced all rows: adults and children are not split into separate models.

In [ ]:
_glance_cols = {'n_images': 'Images', 'Accuracy (overall)': 'Accuracy', 'Sensitivity (overall)': 'Sensitivity (recall)',
                'Specificity (overall)': 'Specificity', 'Precision (overall)': 'Precision', 'F1 (overall)': 'F1',
                'macro_AUROC': 'AUROC (macro)', 'Normal-vs-abnormal accuracy': 'Normal/abnormal accuracy',
                'Normal-vs-abnormal sensitivity': 'Normal/abnormal sensitivity',
                'Normal-vs-abnormal specificity': 'Normal/abnormal specificity', 'pneumonia_AUROC': 'Pneumonia AUROC'}
GL = pd.DataFrame(summary).T
GL = GL[[c for c in _glance_cols if c in GL.columns]].rename(columns=_glance_cols)
for c in GL.columns:
    if c != 'Images': GL[c] = pd.to_numeric(GL[c], errors='coerce').round(4)
GL.to_csv(OUT / 'results_at_a_glance.csv')
print(f'Experiment: {EXP} | model: {" + ".join(b.split(".")[0] for b in CFG["BACKBONES"])} | {LIGHT["parameters_M"]} M parameters')
print(GL.to_string())
fig, ax = plt.subplots(figsize=(min(18, 2 + 1.6 * GL.shape[1]), 0.6 + 0.45 * len(GL))); ax.axis('off')
tb = ax.table(cellText=GL.astype(str).values, rowLabels=GL.index, colLabels=GL.columns, loc='center', cellLoc='center')
tb.auto_set_font_size(False); tb.set_fontsize(9); tb.scale(1, 1.5)
plt.title(f'Results at a glance ({EXP})'); plt.tight_layout(); plt.savefig(FIG / 'results_at_a_glance.png', dpi=140, bbox_inches='tight'); plt.show()

## 11. Subgroup analysis: age bands and sex
**Underdiagnosis rate** = share of images with at least one true finding where the model flags nothing (all calibrated probabilities below their thresholds). **Over-call rate** = share of images with no finding where the model flags something.

In [ ]:
rows_all = te_idx; y_all, m_all = np.nan_to_num(Yraw[rows_all]), ~np.isnan(Yraw[rows_all])
flag = ABN_te.copy(); has = ((y_all == 1) & m_all).any(1)
sub_rows = []
for col in ['age_band', 'sex']:
    vals = np.array([str(v) for v in meta[col].fillna('nan').values[rows_all]], dtype=object)
    for v in sorted(set(vals) - {'nan', 'U', 'None'}):
        k = vals == v
        if k.sum() < 30: continue
        sub_rows.append(dict(attribute=col, group=v, n=int(k.sum()), prevalence_any=float(has[k].mean()),
                             macro_AUROC=macro_auc(y_all[k], P_cal[k], m_all[k].astype(float), min_pos=5),
                             underdiagnosis=float((~flag[k] & has[k]).sum() / max(has[k].sum(), 1)),
                             overcall=float((flag[k] & ~has[k]).sum() / max((~has[k]).sum(), 1))))
SUB = pd.DataFrame(sub_rows, columns=['attribute', 'group', 'n', 'prevalence_any', 'macro_AUROC', 'underdiagnosis', 'overcall'])
SUB.to_csv(OUT / 'subgroups.csv', index=False)
print(SUB.round(3).to_string())
a = SUB[SUB.attribute == 'age_band']
if len(a):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].bar(a.group, a.macro_AUROC, color='#4C72B0'); ax[0].set_ylim(0.5, 1); ax[0].set_title('macro-AUROC by age band')
    ax[1].bar(a.group, a.underdiagnosis, color='#C44E52'); ax[1].set_title('underdiagnosis rate by age band')
    plt.tight_layout(); plt.savefig(FIG / 'subgroups.png', dpi=120); plt.show()

## 12. Shortcut audit
1. **Source probe:** can a linear classifier tell the pediatric source (NIH children vs GWCMC children) from the fused features? High AUROC means the representation encodes the hospital, which can confound pneumonia.
2. **Pneumonia AUROC inside NIH children vs inside GWCMC:** a model that learned "GWCMC look → pneumonia" will be much worse on NIH children.
3. **Age-group head accuracy** (proposed model only).

In [ ]:
ped_rows = np.where(AGEG[te_idx] == 1)[0]
src = (meta.source.values[te_idx][ped_rows] == 'KER').astype(int)
probe = {}
if len(np.unique(src)) == 2 and min(np.bincount(src)) >= 10:
    Zp, grp = pr_te['z'][ped_rows], meta.group.values[te_idx][ped_rows]
    oof = np.zeros(len(ped_rows))
    for tr_, te_ in GroupKFold(5).split(Zp, src, grp):
        clf = LogisticRegression(max_iter=2000, C=0.1).fit(Zp[tr_], src[tr_]); oof[te_] = clf.predict_proba(Zp[te_])[:, 1]
    probe['source_probe_AUROC'] = float(roc_auc_score(src, oof))
if FLAGS['age_head']:
    k = AGEG[te_idx] >= 0
    probe['age_head_AUROC'] = float(roc_auc_score(AGEG[te_idx][k], pr_te['age_logit'][k]))
    probe['age_head_acc'] = float(((pr_te['age_logit'][k] > 0) == (AGEG[te_idx][k] == 1)).mean())
for ts in ['NIH-pediatric', 'Pediatric-GWCMC', 'NIH-adult']:
    if ts in summary: probe[f'pneumonia_AUROC_{ts}'] = summary[ts]['pneumonia_AUROC']
print(json.dumps(probe, indent=2))

## 13. Explainable AI
* Grad-CAM and HiResCAM per branch on the backbone's last feature map, **fused with the image's own gate weights**.
* **Localization** on NIH expert bounding boxes: pointing game (max inside the box), energy gain $\frac{E_{box}/E_{img}}{A_{box}/A_{img}}$ (1 = chance) and IoU of the map thresholded at 50% of its max.
* **Faithfulness:** deletion curve: the most salient pixels are replaced by a blurred copy first, compared with random order; the score is the mean class probability along the curve (lower than random = more faithful).
* **Border-saliency ratio:** share of saliency in the outer 12% frame (text markers, collimation edges). Reported separately for adult and pediatric test images.

In [ ]:
for p_ in model.parameters():
    p_.requires_grad_(True)   # backbones may still be frozen if training stopped early

def blurred(x1):
    # realistic 'removed' content: a heavily blurred copy of the same X-ray (used by deletion, LIME and KernelSHAP)
    k = int(ISZ / 8) | 1
    return torch.stack([torch.from_numpy(cv2.GaussianBlur(c, (k, k), ISZ / 16)) for c in x1.detach().cpu().numpy()]).to(x1.device)

def to_nchw(f, nf):
    return f.permute(0, 3, 1, 2) if (f.ndim == 4 and f.shape[-1] == nf and f.shape[1] != nf) else f

def cams(net, x, cls, age=None, age_known=None, n_smooth=None, max_batch=None):
    # Grad-CAM and HiResCAM on each branch's last feature map, averaged over n_smooth noisy copies (copy 0 is clean)
    n_smooth = max(1, CFG['CAM_SMOOTH'] if n_smooth is None else n_smooth)
    max_batch = max_batch or CFG['CAM_MAX_BATCH']
    B = len(x)
    step = max(1, max_batch // n_smooth)
    if B > step:   # split into chunks so (images x noisy copies) with gradients fits in memory
        parts = [cams(net, x[i:i + step], cls[i:i + step], None if age is None else age[i:i + step],
                      None if age_known is None else age_known[i:i + step], n_smooth, max_batch) for i in range(0, B, step)]
        return {k_: np.concatenate([p_[0][k_] for p_ in parts]) for k_ in parts[0][0]}, np.concatenate([p_[1] for p_ in parts])
    if n_smooth > 1:
        noise = torch.randn((B, n_smooth) + tuple(x.shape[1:]), device=x.device) * 0.1; noise[:, 0] = 0
        x = (x[:, None] + noise).flatten(0, 1); cls = cls.repeat_interleave(n_smooth)
        age = age.repeat_interleave(n_smooth) if age is not None else None
        age_known = age_known.repeat_interleave(n_smooth) if age_known is not None else None
    net.eval(); net.zero_grad(set_to_none=True)
    out = net(x, age, age_known, return_maps=True)
    for m in out['maps']: m.retain_grad()
    out['logits'][torch.arange(len(x)), cls].sum().backward()
    res = {'gradcam': [], 'hirescam': []}
    for m, br in zip(out['maps'], net.branches):
        A, G = to_nchw(m.detach().float(), br.nf), to_nchw(m.grad.detach().float(), br.nf)
        res['gradcam'].append(F.relu((G.mean((2, 3), keepdim=True) * A).sum(1)))
        res['hirescam'].append(F.relu((G * A).sum(1)))
    w = out['gate'].detach().view(B, n_smooth, -1).mean(1)
    final = {}
    for k_, lst in res.items():
        ups = []
        for c in lst:
            c = c.view(B, n_smooth, *c.shape[1:]).mean(1)          # average the noisy copies
            c = F.interpolate(c[:, None], size=(ISZ, ISZ), mode='bilinear', align_corners=False)[:, 0]
            c = c / (c.flatten(1).max(1)[0][:, None, None] + 1e-8); ups.append(c)
        final[k_ + '_branches'] = torch.stack(ups, 1).cpu().numpy()
        final[k_ + '_fused'] = (torch.stack(ups, 1) * w[:, :, None, None]).sum(1).cpu().numpy()
    probs = torch.sigmoid(out['logits'].detach().view(B, n_smooth, -1)[:, 0]).cpu().numpy()
    return final, probs

def batch_x(rows):
    ds = CXRDataset(rows); return torch.stack([torch.from_numpy(ds.load(j)) for j in rows]).to(DEVICE)

def age_t(rows):
    return torch.tensor(np.clip(AGEG[rows], 0, 1), device=DEVICE), torch.tensor((AGEG[rows] >= 0).astype(np.float32), device=DEVICE)

xai = {}
if bbox_csv:
    bb = pd.read_csv(bbox_csv)
    bb.columns = ['img', 'label', 'x', 'y', 'w', 'h'] + list(bb.columns[6:])
    bb['label'] = bb.label.replace({'Infiltrate': 'Infiltration'})
    uid2row = pd.Series(np.arange(len(meta)), index=meta.uid.values)
    bb['row'] = ('NIH_' + bb.img).map(uid2row)
    bb = bb[bb.row.notna() & bb.label.isin(LABELS)].copy(); bb['row'] = bb.row.astype(int)
    bt_ = bb[meta.split.values[bb.row] == 'test']
    bb = bt_ if len(bt_) >= 50 else bb
    print(f'Bounding boxes used: {len(bb)} ({(meta.split.values[bb.row] == "test").mean():.0%} from locked test)')
    recs = []
    for s in range(0, len(bb), 16):
        chunk = bb.iloc[s:s + 16]; rows = chunk.row.values
        a, ak = age_t(rows)
        cm_, _ = cams(model, batch_x(rows), torch.tensor([LABELS.index(l) for l in chunk.label], device=DEVICE), a, ak)
        for i, r in enumerate(chunk.itertuples()):
            sx, sy = ISZ / meta.orig_w.values[r.row], ISZ / meta.orig_h.values[r.row]
            box = np.zeros((ISZ, ISZ), bool)
            box[int(r.y * sy):max(int((r.y + r.h) * sy), int(r.y * sy) + 1), int(r.x * sx):max(int((r.x + r.w) * sx), int(r.x * sx) + 1)] = True
            for meth in ['gradcam_fused', 'hirescam_fused'] + [f'hirescam_b{k}' for k in range(len(CFG['BACKBONES']))]:
                cmap = cm_['hirescam_branches'][i, int(meth[-1])] if '_b' in meth else cm_[meth][i]
                e = cmap.sum() + 1e-8; mx = np.unravel_index(cmap.argmax(), cmap.shape)
                binm = cmap >= 0.5 * cmap.max()
                recs.append(dict(method=meth, label=r.label, hit=bool(box[mx]), energy_gain=(cmap[box].sum() / e) / (box.mean() + 1e-8),
                                 iou=(binm & box).sum() / max((binm | box).sum(), 1)))
    LOC = pd.DataFrame(recs)
    loc_tab = LOC.groupby('method')[['hit', 'energy_gain', 'iou']].mean()
    print(loc_tab.round(3)); print(LOC[LOC.method == 'hirescam_fused'].groupby('label')[['hit', 'energy_gain', 'iou']].mean().round(3))
    LOC.to_csv(OUT / 'xai_localization.csv', index=False); xai['localization'] = loc_tab.round(4).to_dict()

In [ ]:
# Faithfulness (deletion) and border-saliency on adult and pediatric test images
rng = np.random.default_rng(0)
trapz = getattr(np, 'trapezoid', None) or np.trapz
def pick(ts, n, require_pos=True):
    k = np.where(meta.test_set.values == ts)[0]
    if require_pos and len(k):
        k = k[np.nansum(Yraw[k], 1) > 0]
    return rng.choice(k, min(n, len(k)), replace=False) if len(k) else k

frame = np.zeros((ISZ, ISZ), bool); bw = int(0.12 * ISZ); frame[:bw] = frame[-bw:] = True; frame[:, :bw] = frame[:, -bw:] = True
steps = np.linspace(0, 0.5, 11)
for ts in ['NIH-adult', 'NIH-pediatric', 'Pediatric-GWCMC']:
    rows = pick(ts, 48)
    if len(rows) == 0: continue
    cls = np.array([np.nanargmax(np.where(np.isnan(Yraw[r]), -1, Yraw[r])) for r in rows])
    border, del_s, del_r = [], [], []
    for s in range(0, len(rows), 8):
        rr, cc = rows[s:s + 8], cls[s:s + 8]; x = batch_x(rr); a, ak = age_t(rr)
        cm_, _ = cams(model, x, torch.tensor(cc, device=DEVICE), a, ak)
        sal = cm_['hirescam_fused']
        border += list((sal * frame).sum((1, 2)) / (sal.sum((1, 2)) + 1e-8))
        for i in range(len(rr)):
            order_s = np.argsort(-sal[i].ravel()); order_r = rng.permutation(ISZ * ISZ)
            xblur = blurred(x[i])
            for order, store in [(order_s, del_s), (order_r, del_r)]:
                xs = x[i:i + 1].repeat(len(steps), 1, 1, 1).clone()
                for j, fr in enumerate(steps):
                    kill = order[:int(fr * ISZ * ISZ)]
                    xs[j, :, kill // ISZ, kill % ISZ] = xblur[:, kill // ISZ, kill % ISZ]   # deleted pixels -> blurred copy
                with torch.no_grad():
                    p = torch.sigmoid(model(xs, a[i:i + 1].repeat(len(steps)), ak[i:i + 1].repeat(len(steps)))['logits'][:, cc[i]]).cpu().numpy()
                store.append(trapz(p, steps) / steps[-1])   # mean probability along the deletion curve (no division by p0)
    xai[ts] = dict(border_saliency=float(np.mean(border)), deletion_AUC_saliency=float(np.mean(del_s)), deletion_AUC_random=float(np.mean(del_r)),
                   border_area=float(frame.mean()))
print(json.dumps({k: v for k, v in xai.items() if k != 'localization'}, indent=2))

In [ ]:
# Qualitative panel: adult and pediatric examples with branch maps and gate-fused map
def show_examples(ts_list=('NIH-adult', 'NIH-pediatric', 'Pediatric-GWCMC'), n=2):
    sel = [(ts, r) for ts in ts_list for r in pick(ts, n)]
    if not sel: return
    K = len(CFG['BACKBONES'])
    fig, axs = plt.subplots(len(sel), K + 2, figsize=(3 * (K + 2), 3 * len(sel)))
    axs = np.atleast_2d(axs)
    for i, (ts, r) in enumerate(sel):
        a, ak = age_t(np.array([r])); x = batch_x(np.array([r]))
        with torch.no_grad():
            p = torch.sigmoid(model(x, a, ak)['logits'])[0].cpu().numpy()
        c = int(p.argmax()); cm_, _ = cams(model, x, torch.tensor([c], device=DEVICE), a, ak)
        img = cv2.resize(cv2.imread(meta.path.values[r], 0), (ISZ, ISZ))
        truth = [L for L, v in zip(LABELS, Yraw[r]) if v == 1] or ['No finding']
        axs[i, 0].imshow(img, cmap='gray'); axs[i, 0].set_title(f'{ts}\ntrue: {",".join(truth)[:30]}', fontsize=8)
        for k in range(K):
            axs[i, k + 1].imshow(img, cmap='gray'); axs[i, k + 1].imshow(np.ma.masked_less(cm_['hirescam_branches'][0, k], 0.15), cmap='jet', alpha=0.5, vmin=0, vmax=1)
            axs[i, k + 1].set_title(f'{CFG["BACKBONES"][k].split(".")[0]} (gate {pr_gate(r, k):.2f})', fontsize=8)
        axs[i, -1].imshow(img, cmap='gray'); axs[i, -1].imshow(np.ma.masked_less(cm_['hirescam_fused'][0], 0.15), cmap='jet', alpha=0.5, vmin=0, vmax=1)
        axs[i, -1].set_title(f'fused: {LABELS[c]} p={p[c]:.2f}', fontsize=8)
        for ax_ in axs[i]: ax_.axis('off')
    plt.tight_layout(); plt.savefig(FIG / 'xai_examples.png', dpi=110); plt.show()

def pr_gate(r, k):
    pos_ = np.where(te_idx == r)[0]
    return float(pr_te['gate'][pos_[0], k]) if len(pos_) else float('nan')
show_examples()

## 13b. LIME and KernelSHAP (model-agnostic explanations)
Both explain one prediction by switching **superpixels** (SLIC regions) on and off and watching the class probability.
* **LIME** fits a weighted ridge model on random on/off patterns (kernel width 0.25 on cosine distance).
* **KernelSHAP** samples coalitions from the Shapley kernel and fits a linear model whose coefficients approximate Shapley values (empty and full image forced by large weights).

They are scored on the **same** NIH expert-box images as HiResCAM (pointing game, energy gain, IoU), with the time each method needs per image.

In [ ]:
try:
    from skimage.segmentation import slic
except Exception:
    slic = None
from sklearn.linear_model import Ridge, LinearRegression

def superpixels(f, n):
    # Fix: the earlier setting (compactness 0.05, no smoothing) produced ~20 regions with one region covering
    # half of the image, so LIME/SHAP gave all credit to that region. Smoothed SLIC gives ~n balanced regions.
    seg = None
    if slic is not None:
        for comp in (0.1, 0.3, 1.0):
            try:
                cand = slic(f.astype(np.float64), n_segments=n, compactness=comp, sigma=2, channel_axis=None, start_label=0)
            except Exception:
                break
            _, cnts = np.unique(cand, return_counts=True)
            if len(cnts) >= n // 2 and cnts.max() <= 0.06 * f.size:   # enough regions, none too large
                seg = cand; break
    if seg is None:   # fallback: regular grid
        g = max(2, int(round(math.sqrt(n)))); idx = np.minimum(np.arange(ISZ) * g // ISZ, g - 1)
        seg = idx[:, None] * g + idx[None, :]
    return np.unique(seg, return_inverse=True)[1].reshape(seg.shape)

@torch.no_grad()
def masked_probs(x1, seg_t, keep, cls, a, ak, bs=64):
    # returns the class LOGIT (not the probability): sigmoid saturation flattened the earlier LIME/SHAP fits
    out, xb0 = [], blurred(x1)
    for s in range(0, len(keep), bs):
        mk = torch.tensor(keep[s:s + bs], device=DEVICE, dtype=torch.float32)[:, seg_t][:, None]
        xb = x1[None] * mk + xb0[None] * (1 - mk)
        n_ = len(xb)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            lo = model(xb, a.expand(n_), ak.expand(n_))['logits'][:, cls]
        out.append(lo.float().cpu())
    return torch.cat(out).numpy()

def lime_map(x1, seg, cls, a, ak, n_samples, rng):
    S = int(seg.max()) + 1
    Z = rng.random((n_samples, S)) > 0.5; Z[0] = True
    p = masked_probs(x1, torch.tensor(seg, device=DEVICE), Z, cls, a, ak)
    d = 1 - np.sqrt(Z.sum(1) / S)                        # cosine distance to the full image
    w = np.exp(-(d ** 2) / 0.25 ** 2)
    coef = Ridge(alpha=1.0).fit(Z.astype(float), p, sample_weight=w).coef_
    return coef[seg]

def kernel_shap_map(x1, seg, cls, a, ak, n_samples, rng):
    S = int(seg.max()) + 1
    ks = np.arange(1, S); pk = (S - 1) / (ks * (S - ks)); pk = pk / pk.sum()
    sizes = rng.choice(ks, n_samples, p=pk)
    Z = np.zeros((n_samples + 2, S), bool)
    for i, k_ in enumerate(sizes):
        Z[i, rng.choice(S, k_, replace=False)] = True
    Z[-1] = True                                        # full image (row -2 stays empty)
    p = masked_probs(x1, torch.tensor(seg, device=DEVICE), Z, cls, a, ak)
    w = np.ones(len(Z)); w[-2:] = 1e6
    coef = LinearRegression().fit(Z.astype(float), p, sample_weight=w).coef_
    return coef[seg]

def show_attr(ax, f, m_, seg=None, top=8):
    # positive evidence only, weak values transparent so the X-ray stays visible; for LIME/SHAP the top regions are outlined
    pos = np.clip(m_, 0, None); pos = pos / (pos.max() + 1e-8)
    ax.imshow(f, cmap='gray')
    ax.imshow(np.ma.masked_less(pos, 0.15), cmap='jet', alpha=0.5, vmin=0, vmax=1)
    if seg is not None:
        seg_score = np.array([m_[seg == k].mean() for k in range(int(seg.max()) + 1)])
        best = [k for k in np.argsort(-seg_score)[:top] if seg_score[k] > 0]
        if best:
            ax.contour(np.isin(seg, best).astype(float), levels=[0.5], colors='yellow', linewidths=1.2)

def box_mask(row, bx, by, bw_, bh_):
    sx, sy = ISZ / meta.orig_w.values[row], ISZ / meta.orig_h.values[row]
    m = np.zeros((ISZ, ISZ), bool)
    m[int(by * sy):max(int((by + bh_) * sy), int(by * sy) + 1), int(bx * sx):max(int((bx + bw_) * sx), int(bx * sx) + 1)] = True
    return m

def loc_scores(sal, box):
    sal = np.clip(sal, 0, None)
    if sal.max() <= 0:
        return dict(hit=False, energy_gain=0.0, iou=0.0)
    mx = np.unravel_index(sal.argmax(), sal.shape); binm = sal >= 0.5 * sal.max()
    return dict(hit=bool(box[mx]), energy_gain=float((sal[box].sum() / (sal.sum() + 1e-8)) / (box.mean() + 1e-8)),
                iou=float((binm & box).sum() / max((binm | box).sum(), 1)))

rng_l = np.random.default_rng(1)
lime_recs, lime_shap = [], {}
if bbox_csv and 'bb' in globals() and len(bb):
    sel = bb.drop_duplicates('row').head(CFG['LIME_IMAGES'])
    for r in sel.itertuples():
        rows = np.array([r.row]); x = batch_x(rows); a, ak = age_t(rows); c = LABELS.index(r.label)
        f = x[0, 0].detach().cpu().numpy() * float(STD[0, 0, 0]) + float(MEAN[0, 0, 0])
        seg = superpixels(f, CFG['N_SUPERPIXELS']); box = box_mask(r.row, r.x, r.y, r.w, r.h)
        t0 = time.time(); cm_, _ = cams(model, x, torch.tensor([c], device=DEVICE), a, ak); t_cam = time.time() - t0
        t0 = time.time(); lm = lime_map(x[0], seg, c, a, ak, CFG['LIME_SAMPLES'], rng_l); t_lime = time.time() - t0
        t0 = time.time(); sh = kernel_shap_map(x[0], seg, c, a, ak, CFG['SHAP_SAMPLES'], rng_l); t_shap = time.time() - t0
        for name, m_, t_ in [('HiResCAM-fused', cm_['hirescam_fused'][0], t_cam), ('LIME', lm, t_lime), ('KernelSHAP', sh, t_shap)]:
            lime_recs.append(dict(method=name, label=r.label, seconds=t_, **loc_scores(m_, box)))
    LS = pd.DataFrame(lime_recs)
    if len(LS):
        tab = LS.groupby('method')[['hit', 'energy_gain', 'iou', 'seconds']].mean()
        print(f'{sel.shape[0]} expert-box images'); print(tab.round(3))
        LS.to_csv(OUT / 'xai_lime_shap.csv', index=False); lime_shap = tab.round(4).to_dict()
else:
    print('No NIH bounding boxes available: LIME/KernelSHAP localization skipped.')

# qualitative panel: HiResCAM vs LIME vs KernelSHAP, adult and pediatric
panel = [(ts, r_) for ts in ['NIH-adult', 'NIH-pediatric', 'Pediatric-GWCMC'] for r_ in pick(ts, 1)] if 'pick' in globals() else []
if panel:
    fig, axs = plt.subplots(len(panel), 4, figsize=(13, 3.3 * len(panel))); axs = np.atleast_2d(axs)
    for i, (ts, r_) in enumerate(panel):
        rows = np.array([r_]); x = batch_x(rows); a, ak = age_t(rows)
        c = int(np.nanargmax(np.where(np.isnan(Yraw[r_]), -1, Yraw[r_])))
        f = x[0, 0].detach().cpu().numpy() * float(STD[0, 0, 0]) + float(MEAN[0, 0, 0]); seg = superpixels(f, CFG['N_SUPERPIXELS'])
        cm_, pr_ = cams(model, x, torch.tensor([c], device=DEVICE), a, ak)
        maps_ = [cm_['hirescam_fused'][0], lime_map(x[0], seg, c, a, ak, CFG['LIME_SAMPLES'], rng_l), kernel_shap_map(x[0], seg, c, a, ak, CFG['SHAP_SAMPLES'], rng_l)]
        axs[i, 0].imshow(f, cmap='gray'); axs[i, 0].set_title(f'{ts}\n{LABELS[c]} p={pr_[0, c]:.2f}', fontsize=9)
        for j, (nm, m_) in enumerate(zip(['HiResCAM (gate-fused)', 'LIME (top 8 regions outlined)', 'KernelSHAP (top 8 regions outlined)'], maps_)):
            show_attr(axs[i, j + 1], f, m_, None if j == 0 else seg); axs[i, j + 1].set_title(nm, fontsize=9)
        for ax_ in axs[i]: ax_.axis('off')
    plt.tight_layout(); plt.savefig(FIG / 'xai_lime_shap_examples.png', dpi=110); plt.show()

## 14. Robustness to image corruptions (three severities)

In [ ]:
rob = []
rob_rows = {ts: pick(ts, 1500 if not CFG['DEBUG'] else 100, require_pos=False) for ts in ['NIH-adult', 'Pediatric-GWCMC']}
rob_rows = {k: np.sort(v) for k, v in rob_rows.items() if len(v)}
for ts, rows in rob_rows.items():
    yb, mb = np.nan_to_num(Yraw[rows]), (~np.isnan(Yraw[rows])).astype(float)
    base = macro_auc(yb, 1 / (1 + np.exp(-predict(model, rows)['logits'])), mb)
    rob.append(dict(test_set=ts, corruption='none', severity=0, macro_AUROC=base))
    for kind in ['noise', 'blur', 'contrast', 'jpeg']:
        for sev in range(3):
            pr_ = predict(model, rows, corruption=(kind, sev))
            rob.append(dict(test_set=ts, corruption=kind, severity=sev + 1, macro_AUROC=macro_auc(yb, 1 / (1 + np.exp(-pr_['logits'])), mb)))
ROB = pd.DataFrame(rob); ROB.to_csv(OUT / 'robustness.csv', index=False)
print(ROB.pivot_table(index=['test_set', 'corruption'], columns='severity', values='macro_AUROC').round(3))

## 14b. Few-shot adaptation to a pediatric site
How much does a handful of labelled children improve the frozen model on a pediatric site? For each target test set and label, *k* positive and *k* negative **patients** form the support set; all other patients of that site form the query set (no patient in both).

* **Zero adaptation:** the frozen model's logit $\ell_0(x)$.
* **Linear probe:** logistic regression on the fused features *z* only (ignores the trained head).
* **Prior-anchored adapter (proposed, as in the FYDP):** $s(x)=\ell_0(x)+w^\top \tilde z(x)+b$ with an L2 penalty $\lambda\lVert w\rVert^2$. With $w=0$ it is exactly the frozen model. $\lambda$, including *no adaptation*, is chosen by 3-fold grouped CV on the support set only. Features are standardised with unlabelled target images.

Each *k* is repeated `FEWSHOT_REPS` times with different support patients. This is most informative for the `adult_only` experiment (adult model meets children).

In [ ]:
from sklearn.metrics import balanced_accuracy_score

def fit_adapter(Zs, l0, y, lam):
    Zt, l0t, yt = torch.tensor(Zs, dtype=torch.float32), torch.tensor(l0, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)
    w = torch.zeros(Zt.shape[1], requires_grad=True); b0 = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([w, b0], lr=0.5, max_iter=100, line_search_fn='strong_wolfe')
    def clo():
        opt.zero_grad(); l = F.binary_cross_entropy_with_logits(l0t + Zt @ w + b0, yt) + lam * (w * w).sum(); l.backward(); return l
    opt.step(clo)
    return w.detach().numpy(), float(b0.item())

def bce_np(s, y):
    return float(F.binary_cross_entropy_with_logits(torch.tensor(s, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)))

def choose_lam(Zs, l0, y, groups, grid=(0.01, 0.1, 1.0, 10.0)):
    if len(np.unique(groups)) < 6:
        return 1.0
    scores = {lam: [] for lam in grid}; scores[None] = []
    for tr_, va_ in GroupKFold(3).split(Zs, y, groups):
        if len(np.unique(y[tr_])) < 2: continue
        scores[None].append(bce_np(l0[va_], y[va_]))
        for lam in grid:
            w, b_ = fit_adapter(Zs[tr_], l0[tr_], y[tr_], lam); scores[lam].append(bce_np(l0[va_] + Zs[va_] @ w + b_, y[va_]))
    if not scores[None]:
        return 1.0
    return min(scores, key=lambda k_: np.mean(scores[k_]))      # None = keep the frozen model

FS = []
for target in [t for t in CFG['FEWSHOT_TARGETS'] if t in TEST_SETS]:
    kk = np.where(meta.test_set.values[te_idx] == target)[0]
    Zt = pr_te['z'][kk]; Zt = (Zt - Zt.mean(0)) / (Zt.std(0) + 1e-6)
    L0 = pr_te['logits'][kk]; pids = meta.patient_id.values[te_idx][kk]
    for c in range(NL):
        yc = Yraw[te_idx[kk], c]; known = ~np.isnan(yc)
        if known.sum() < 20: continue
        pat = pd.Series(yc[known], index=pids[known]).groupby(level=0).max()
        pos_p, neg_p = pat.index[pat == 1].values, pat.index[pat == 0].values
        for k in CFG['FEWSHOT_KS']:
            if len(pos_p) < k + 5 or len(neg_p) < k + 5: continue
            for rep in range(CFG['FEWSHOT_REPS']):
                rs_ = np.random.default_rng(1000 * rep + k)
                sp = set(rs_.choice(pos_p, k, replace=False)) | set(rs_.choice(neg_p, k, replace=False))
                sup = known & np.isin(pids, list(sp)); qry = known & ~np.isin(pids, list(sp))
                ys, yq = yc[sup], yc[qry]
                if len(np.unique(yq)) < 2 or len(np.unique(ys)) < 2: continue
                res_ = {'zero-adaptation': L0[qry, c]}
                res_['linear probe'] = LogisticRegression(C=1.0, max_iter=2000).fit(Zt[sup], ys).decision_function(Zt[qry])
                lam = choose_lam(Zt[sup], L0[sup, c], ys, pids[sup])
                if lam is None:
                    res_['prior-anchored adapter'] = L0[qry, c]
                else:
                    w, b_ = fit_adapter(Zt[sup], L0[sup, c], ys, lam); res_['prior-anchored adapter'] = L0[qry, c] + Zt[qry] @ w + b_
                for mth, sc in res_.items():
                    FS.append(dict(target=target, label=LABELS[c], k=k, rep=rep, method=mth, lam=lam if mth == 'prior-anchored adapter' else np.nan,
                                   n_query=int(qry.sum()), AUROC=roc_auc_score(yq, sc), BA=balanced_accuracy_score(yq, sc > 0)))
FSdf = pd.DataFrame(FS)
fewshot = {}
if len(FSdf):
    FSdf.to_csv(OUT / 'fewshot.csv', index=False)
    agg = FSdf.groupby(['target', 'label', 'k', 'method'])[['AUROC', 'BA']].agg(['mean', 'std']).round(3)
    print(agg.to_string())
    focus = FSdf[FSdf.label == 'Pneumonia'] if (FSdf.label == 'Pneumonia').any() else FSdf
    tg = focus.target.unique()
    fig, axs = plt.subplots(1, len(tg), figsize=(6 * len(tg), 4), squeeze=False)
    for ax_, t in zip(axs[0], tg):
        d_ = focus[focus.target == t].groupby(['method', 'k']).BA.agg(['mean', 'std']).reset_index()
        for mth, g_ in d_.groupby('method'):
            ax_.errorbar(g_.k, g_['mean'], yerr=g_['std'], marker='o', capsize=3, label=mth)
        ax_.set_xscale('log'); ax_.set_xlabel('labelled patients per class (k)'); ax_.set_ylabel('balanced accuracy')
        ax_.set_title(f'{t}: {focus.label.iloc[0]}'); ax_.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(FIG / 'fewshot.png', dpi=120); plt.show()
    fewshot = {f'{t}|{l}|k={k}|{m}': float(v) for (t, l, k, m), v in FSdf.groupby(['target', 'label', 'k', 'method']).BA.mean().items()}
else:
    print('Not enough labelled patients in the target sets for few-shot experiments.')

## 15. Save the summary of this experiment

In [ ]:
res = dict(experiment=EXP, flags=FLAGS, config=CFG, summary=summary, shortcut_probe=probe, model_size=LIGHT,
           xai={k: v for k, v in xai.items()}, lime_kernelshap=globals().get('lime_shap', {}),
           fewshot=globals().get('fewshot', {}), history=hist.to_dict('records'))
json.dump(res, open(OUT / 'summary.json', 'w'), indent=2, default=float)
DEPLOY_SD = {k: v for k, v in model.state_dict().items() if not k.startswith('src_head')}   # training-only head dropped
torch.save({'state_dict': DEPLOY_SD, 'CFG': CFG, 'FLAGS': FLAGS, 'LABELS': LABELS, 'TEMP': TEMP, 'THR': THR, 'ABN_THR': ABN_THR,
            'PRIOR_LOGIT': PRIOR_LOGIT}, OUT / 'deploy_bundle.pt')
print('Saved to', OUT)
_cols = ['n_images', 'macro_AUROC', 'CI95', 'Accuracy (overall)', 'Sensitivity (overall)', 'Specificity (overall)', 'Precision (overall)', 'F1 (overall)',
         'Normal-vs-abnormal accuracy', 'Normal-vs-abnormal sensitivity', 'pneumonia_AUROC']
_S = pd.DataFrame(summary).T
print(_S[[c for c in _cols if c in _S.columns]].to_string())

## 16. Inference on a new image (adult or child), one model for both
The same trained model is used for every patient; there is no separate adult or child model. `age` is optional. When it is given the model uses the age-specific prevalence and thresholds; otherwise it uses its own predicted age group.

In [ ]:
def predict_image(path, age=None, show=True):
    raw = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(raw, (S, S), interpolation=cv2.INTER_AREA)
    f = cv2.resize(img, (ISZ, ISZ), interpolation=cv2.INTER_AREA).astype(np.float32) / 255
    x = torch.from_numpy(((np.repeat(f[None], 3, 0) - MEAN) / STD).astype(np.float32))[None].to(DEVICE)
    if age is not None:
        a, ak = torch.tensor([float(age < 18)], device=DEVICE), torch.tensor([1.0], device=DEVICE)
    else:
        a = ak = None
    with torch.no_grad():
        o = model(x, a, ak)
    ped_p = float(torch.sigmoid(o['age_logit'])[0]) if o['age_logit'] is not None else None
    grp = group_of_age(age, ped_p)
    p, thr = calibrate(o['logits'].float().cpu().numpy(), np.array([grp]))
    res = pd.DataFrame({'probability': p[0], 'threshold': thr[0], 'flag': p[0] >= thr[0]}, index=LABELS).sort_values('probability', ascending=False)
    res.attrs['abnormal'] = bool(abnormal(p, thr, np.array([grp]))[0]); res.attrs['group'] = GNAMES[grp]
    if show:
        c = LABELS.index(res.index[0]); cm_, _ = cams(model, x, torch.tensor([c], device=DEVICE), a, ak)
        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        ax[0].imshow(f, cmap='gray'); ax[0].set_title('input' + (f' (age {age})' if age is not None else '')); ax[0].axis('off')
        ax[1].imshow(f, cmap='gray'); ax[1].imshow(cm_['hirescam_fused'][0], cmap='jet', alpha=0.4)
        ax[1].set_title(f'{res.index[0]}: {res.probability.iloc[0]:.2f}'); ax[1].axis('off'); plt.show()
    return res

# The SAME model analyses an adult and a child X-ray
for ts_, age_ in [('NIH-adult', 55), ('Pediatric-GWCMC', 4), ('NIH-pediatric', 10)]:
    d_ = meta[meta.test_set == ts_]
    if len(d_):
        r_ = d_.iloc[0]; a_txt = int(r_.age) if not np.isnan(r_.age) else age_
        truth = [L for L in LABELS if r_[L] == 1] or ['No finding']
        print(f'\n=== {ts_} (age {a_txt}) | true: {", ".join(truth)} ===')
        out_ = predict_image(r_.path, age=a_txt)
        print(f"calibration group: {out_.attrs['group']} | image-level decision: {'ABNORMAL' if out_.attrs['abnormal'] else 'normal'}")
        print(out_.head(5).round(3))

## 16b. Web application (English + বাংলা)
The cell writes a self-contained **Gradio** app to `results/<experiment>/webapp/`:
* `app.py`: model code, preprocessing, age-specific calibration, gate-fused HiResCAM heatmap, bacterial/viral subtype for children, bilingual output and a medical disclaimer;
* `deploy_bundle_fp16.pt`: weights in half precision (about half the size) + temperatures + thresholds;
* `requirements.txt` and `README.md` ready for **Hugging Face Spaces** (create a Gradio Space, upload the four files).

The app is first tested on a test image without the user interface. Set `LAUNCH_WEBAPP=True` to open it from Kaggle with a temporary public link.

In [ ]:
import shutil, importlib
APP_CODE = r'''# AGF-Net web app: multi-disease chest X-ray analysis for adults and children (research use only)
import os
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F, timm

BUNDLE = os.environ.get('AGF_BUNDLE', os.path.join(os.path.dirname(os.path.abspath(__file__)), 'deploy_bundle_fp16.pt'))
MEAN = np.array([0.485, 0.456, 0.406], np.float32)[:, None, None]
STD = np.array([0.229, 0.224, 0.225], np.float32)[:, None, None]
BN = {'Atelectasis': 'অ্যাটেলেক্টাসিস (ফুসফুস চুপসে যাওয়া)', 'Cardiomegaly': 'কার্ডিওমেগালি (হৃৎপিণ্ড বড়)',
      'Effusion': 'প্লুরাল ইফিউশন (ফুসফুসের চারপাশে তরল)', 'Infiltration': 'ইনফিল্ট্রেশন', 'Mass': 'মাস (পিণ্ড)',
      'Nodule': 'নডিউল', 'Pneumonia': 'নিউমোনিয়া', 'Pneumothorax': 'নিউমোথোরাক্স', 'Consolidation': 'কনসোলিডেশন',
      'Edema': 'ইডিমা (ফুসফুসে তরল)', 'Emphysema': 'এমফাইসেমা', 'Fibrosis': 'ফাইব্রোসিস',
      'Pleural_Thickening': 'প্লুরা পুরু হওয়া', 'Hernia': 'হার্নিয়া'}
DISCLAIMER_EN = '**Research prototype, not a medical diagnosis.** Always consult a qualified doctor or radiologist.'
DISCLAIMER_BN = '**এটি গবেষণার জন্য তৈরি একটি প্রোটোটাইপ, চিকিৎসা নির্ণয় নয়।** সবসময় একজন যোগ্য চিকিৎসক বা রেডিওলজিস্টের পরামর্শ নিন।'


class Branch(nn.Module):
    def __init__(self, name, img_size):
        super().__init__()
        kw = {'img_size': img_size} if (img_size != 224 and any(k in name for k in ('swin', 'vit', 'eva'))) else {}
        self.net = timm.create_model(name, pretrained=False, num_classes=0, **kw)
        self.nf = self.net.num_features

    def forward(self, x):
        f = self.net.forward_features(x)
        return f, self.net.forward_head(f, pre_logits=True)


class AGFNet(nn.Module):
    def __init__(self, names, n_labels, proj, dropout, film, age_head, sub_head, logit_adj, prior_logit, tau, img_size):
        super().__init__()
        self.branches = nn.ModuleList([Branch(n, img_size) for n in names])
        self.projs = nn.ModuleList([nn.Sequential(nn.Linear(b.nf, proj), nn.LayerNorm(proj), nn.GELU()) for b in self.branches])
        K = len(names)
        self.gate = nn.Sequential(nn.Linear(proj * K, 256), nn.GELU(), nn.Linear(256, K)) if K > 1 else None
        self.age_head = nn.Linear(proj, 1) if age_head else None
        self.film = nn.Sequential(nn.Linear(1, 64), nn.GELU(), nn.Linear(64, 2 * proj)) if film else None
        self.head = nn.Sequential(nn.LayerNorm(proj), nn.Dropout(dropout), nn.Linear(proj, n_labels))
        self.sub_head = nn.Linear(proj, 2) if sub_head else None
        self.logit_adj, self.tau = logit_adj, tau
        self.register_buffer('prior_logit', torch.tensor(np.asarray(prior_logit), dtype=torch.float32))

    def forward(self, x, age=None, age_known=None, return_maps=False):
        maps, feats = [], []
        for b, pj in zip(self.branches, self.projs):
            f, p = b(x); maps.append(f); feats.append(pj(p))
        Fs = torch.stack(feats, 1)
        w = torch.softmax(self.gate(Fs.flatten(1)), -1) if self.gate is not None else torch.ones(Fs.shape[:2], device=x.device)
        z = (w.unsqueeze(-1) * Fs).sum(1)
        age_logit = self.age_head(z).squeeze(-1) if self.age_head is not None else None
        a = None
        if self.film is not None or self.logit_adj:
            a = torch.sigmoid(age_logit).detach() if age_logit is not None else torch.full((x.shape[0],), 0.5, device=x.device)
            if age is not None:
                a = torch.where(age_known > 0.5, age.float(), a)
        zc = z
        if self.film is not None:
            g, bb = self.film(a.unsqueeze(-1)).chunk(2, -1)
            zc = z * (1 + g) + bb
        logits = self.head(zc)
        if self.logit_adj:
            logits = logits + self.tau * ((1 - a)[:, None] * self.prior_logit[0] + a[:, None] * self.prior_logit[1])
        out = dict(logits=logits, age_logit=age_logit, gate=w, sub_logits=self.sub_head(zc) if self.sub_head is not None else None)
        if return_maps:
            out['maps'] = maps
        return out


_STATE = {}


def get_model():
    if 'm' not in _STATE:
        ck = torch.load(BUNDLE, map_location='cpu', weights_only=False)
        cfg, fl = ck['CFG'], ck['FLAGS']
        m = AGFNet(cfg['BACKBONES'], len(ck['LABELS']), cfg['PROJ_DIM'], cfg['DROPOUT'], fl['film'], fl['age_head'], fl['sub_head'],
                   fl['logit_adj'], ck['PRIOR_LOGIT'], cfg['LOGIT_ADJ_TAU'], cfg['IMG_SIZE'])
        m.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in ck['state_dict'].items()})
        m.eval()
        for p in m.parameters():
            p.requires_grad_(True)
        _STATE.update(m=m, ck=ck)
    return _STATE['m'], _STATE['ck']


def preprocess(image, cfg):
    img = np.asarray(image)
    if img.ndim == 3:
        img = cv2.cvtColor(img[..., :3].astype(np.uint8), cv2.COLOR_RGB2GRAY)
    img = img.astype(np.uint8)
    if cfg.get('USE_CLAHE'):
        img = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)
    s, isz = cfg['CACHE_SIZE'], cfg['IMG_SIZE']
    img = cv2.resize(img, (s, s), interpolation=cv2.INTER_AREA)
    f = cv2.resize(img, (isz, isz), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
    x = torch.from_numpy(((np.repeat(f[None], 3, 0) - MEAN) / STD).astype(np.float32))[None]
    return x, f


def _nchw(t, nf):
    return t.permute(0, 3, 1, 2) if (t.ndim == 4 and t.shape[-1] == nf and t.shape[1] != nf) else t


def analyse(image, age=None):
    m, ck = get_model(); cfg, labels = ck['CFG'], list(ck['LABELS'])
    x, f = preprocess(image, cfg)
    if age is not None and age >= 0:
        a, ak = torch.tensor([float(age < 18)]), torch.tensor([1.0])
    else:
        a = ak = None
    m.zero_grad(set_to_none=True)
    out = m(x, a, ak, return_maps=True)
    for t in out['maps']:
        t.retain_grad()
    logits = out['logits'][0]
    ped_p = float(torch.sigmoid(out['age_logit'][0]).item()) if out['age_logit'] is not None else None
    # calibration group: 0 adult, 1 child 6-17, 2 child 0-5, 3 child of unknown age
    if age is not None and age >= 0:
        cg = 2 if age < 6 else (1 if age < 18 else 0)
    else:
        cg = 3 if (ped_p is not None and ped_p > 0.5) else 0
    TEMP_, THR_ = np.asarray(ck['TEMP']), np.asarray(ck['THR'])
    if TEMP_.shape[0] == 2:                      # older two-group bundles
        cg = 0 if cg == 0 else 1
    g = 0 if cg == 0 else 1
    lg = logits.detach().numpy()
    p = 1 / (1 + np.exp(-lg / TEMP_[cg]))
    thr = THR_[cg]
    if 'ABN_THR' in ck:
        lgt_ = lambda v: np.log(np.clip(v, 1e-6, 1 - 1e-6) / (1 - np.clip(v, 1e-6, 1 - 1e-6)))
        is_abn = bool((lgt_(p) - lgt_(thr)).max() >= np.asarray(ck['ABN_THR'])[g])
    else:
        is_abn = bool((p >= thr).any())
    c = int(np.argmax(p))
    logits[c].backward()
    isz = cfg['IMG_SIZE']; cam = np.zeros((isz, isz), np.float32); gw = out['gate'][0].detach().numpy()
    for k, (t, br) in enumerate(zip(out['maps'], m.branches)):
        A, G = _nchw(t.detach(), br.nf), _nchw(t.grad.detach(), br.nf)
        h = F.relu((A * G).sum(1))[0].numpy()
        h = cv2.resize(h, (isz, isz)); cam += gw[k] * h / (h.max() + 1e-8)
    cam = cam / (cam.max() + 1e-8)
    base = cv2.cvtColor((f * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    heat = cv2.cvtColor(cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    overlay = (0.6 * base + 0.4 * heat).astype(np.uint8)
    sub = None
    if out['sub_logits'] is not None and g == 1 and 'Pneumonia' in labels:
        j = labels.index('Pneumonia')
        if p[j] >= thr[j]:
            sp = torch.softmax(out['sub_logits'][0].detach(), -1).numpy()
            sub = {'bacterial': float(sp[0]), 'viral': float(sp[1])}
    age_prob = float(torch.sigmoid(out['age_logit'][0]).item()) if out['age_logit'] is not None else None
    return dict(labels=labels, prob=p, thr=thr, group='pediatric' if g == 1 else 'adult', age_prob=age_prob,
                top=labels[c], overlay=overlay, subtype=sub, abnormal=is_abn)


def build_demo():
    import gradio as gr

    def name(l, lang):
        return f'{l} ({BN.get(l, l)})' if lang == 'বাংলা' else l

    def run(img, age, lang):
        if img is None:
            return None, None, None, 'Please upload a chest X-ray. / একটি বুকের এক্স-রে আপলোড করুন।'
        age = None if age in (None, '') else float(age)
        r = analyse(img, age)
        order = np.argsort(-r['prob'])
        lab = {name(r['labels'][i], lang): float(r['prob'][i]) for i in order}
        rows = [[name(r['labels'][i], lang), round(float(r['prob'][i]), 3), round(float(r['thr'][i]), 3),
                 ('হ্যাঁ' if lang == 'বাংলা' else 'Yes') if r['prob'][i] >= r['thr'][i] else ('না' if lang == 'বাংলা' else 'No')] for i in order]
        flagged = [name(r['labels'][i], lang) for i in order if r['prob'][i] >= r['thr'][i]]
        if lang == 'বাংলা':
            txt = f"**সামগ্রিক সিদ্ধান্ত:** {'অস্বাভাবিক' if r['abnormal'] else 'স্বাভাবিক'}\n\n"
            txt += f"**বয়স-গ্রুপ:** {'শিশু' if r['group'] == 'pediatric' else 'প্রাপ্তবয়স্ক'}\n\n"
            txt += '**সম্ভাব্য অস্বাভাবিকতা:** ' + (', '.join(flagged) if flagged else 'কোনোটি সীমা অতিক্রম করেনি') + '\n\n'
            if r['subtype']:
                txt += f"**নিউমোনিয়ার ধরন:** ব্যাকটেরিয়াল {r['subtype']['bacterial']:.0%}, ভাইরাল {r['subtype']['viral']:.0%}\n\n"
            txt += f"হিটম্যাপটি দেখায় মডেল **{name(r['top'], lang)}** সিদ্ধান্তের জন্য ছবির কোন অংশ ব্যবহার করেছে।\n\n" + DISCLAIMER_BN
        else:
            txt = f"**Overall:** {'ABNORMAL' if r['abnormal'] else 'normal'}\n\n"
            txt += f"**Age group used:** {r['group']}" + ('' if age is not None else ' (predicted from the image)') + '\n\n'
            txt += '**Findings above their threshold:** ' + (', '.join(flagged) if flagged else 'none') + '\n\n'
            if r['subtype']:
                txt += f"**Pneumonia subtype:** bacterial {r['subtype']['bacterial']:.0%}, viral {r['subtype']['viral']:.0%}\n\n"
            txt += f"The heatmap shows which regions drove the **{r['top']}** score.\n\n" + DISCLAIMER_EN
        return lab, rows, r['overlay'], txt

    with gr.Blocks(title='AGF-Net Chest X-ray (Adult + Pediatric)') as demo:
        gr.Markdown('# AGF-Net: Multi-Disease Chest X-Ray Analysis for Adults and Children\n'
                    '### বড়দের ও শিশুদের বুকের এক্স-রে থেকে একাধিক রোগ শনাক্তকরণ\n\n' + DISCLAIMER_EN + '<br>' + DISCLAIMER_BN)
        with gr.Row():
            with gr.Column():
                img = gr.Image(type='numpy', label='Chest X-ray / বুকের এক্স-রে')
                age = gr.Number(label='Age in years, optional / বয়স (বছর, ঐচ্ছিক)', value=None)
                lang = gr.Radio(['English', 'বাংলা'], value='English', label='Language / ভাষা')
                btn = gr.Button('Analyse / বিশ্লেষণ করুন', variant='primary')
            with gr.Column():
                lab = gr.Label(num_top_classes=5, label='Top findings / প্রধান ফলাফল')
                heat = gr.Image(label='Explanation heatmap / ব্যাখ্যা হিটম্যাপ')
                summ = gr.Markdown()
        table = gr.Dataframe(headers=['Finding', 'Probability', 'Threshold', 'Flagged'], label='All 14 findings / ১৪টি ফলাফল')
        btn.click(run, [img, age, lang], [lab, table, heat, summ])
    return demo


if __name__ == '__main__':
    build_demo().launch()
'''
APP_DIR = OUT / 'webapp'; APP_DIR.mkdir(exist_ok=True)
(APP_DIR / 'app.py').write_text(APP_CODE, encoding='utf-8')
sd16 = {k: (v.detach().cpu().half() if v.is_floating_point() else v.detach().cpu()) for k, v in model.state_dict().items()
        if not k.startswith('src_head')}
torch.save({'state_dict': sd16, 'CFG': CFG, 'FLAGS': FLAGS, 'LABELS': LABELS, 'TEMP': TEMP, 'THR': THR, 'ABN_THR': ABN_THR,
            'PRIOR_LOGIT': PRIOR_LOGIT},
           APP_DIR / 'deploy_bundle_fp16.pt')
(APP_DIR / 'requirements.txt').write_text('torch\ntimm\nopencv-python-headless\nnumpy\ngradio\n')
(APP_DIR / 'README.md').write_text('---\ntitle: AGF-Net Chest X-ray\ncolorFrom: blue\ncolorTo: green\nsdk: gradio\napp_file: app.py\npinned: false\n---\n\n'
                                   'Multi-disease chest X-ray analysis for adults and children (research prototype, not a diagnosis).\n', encoding='utf-8')
print('Web app files:', sorted(p.name for p in APP_DIR.iterdir()), f"| bundle {(APP_DIR / 'deploy_bundle_fp16.pt').stat().st_size / 1e6:.0f} MB")

# self-test: the app code must reproduce the notebook model on a test image
os.environ['AGF_BUNDLE'] = str(APP_DIR / 'deploy_bundle_fp16.pt')
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
webapp = importlib.reload(sys.modules['app']) if 'app' in sys.modules else importlib.import_module('app')
webapp._STATE.clear()
r0 = int(te_idx[0]); raw0 = cv2.imread(meta.path.values[r0], cv2.IMREAD_GRAYSCALE)
age0 = 4.0 if AGEG[r0] == 1 else 50.0
res_app = webapp.analyse(raw0, age0)
ref = predict_image(meta.path.values[r0], age=age0, show=False)
diff = np.abs(ref.loc[LABELS, 'probability'].values - res_app['prob']).max()
print(f'App vs notebook max probability difference (fp16 weights): {diff:.4f} | top finding: {res_app["top"]} | group: {res_app["group"]}')
plt.figure(figsize=(4, 4)); plt.imshow(res_app['overlay']); plt.axis('off'); plt.title('Web app heatmap'); plt.show()
shutil.make_archive(str(OUT / 'webapp'), 'zip', APP_DIR)
print('Download:', OUT / 'webapp.zip')

if CFG['LAUNCH_WEBAPP']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'], check=False)
    webapp.build_demo().launch(share=True)   # prints a public *.gradio.live link valid while the session runs

## 17. Compare experiments (run after several experiments)
Attach or copy earlier `results/<experiment>/` folders into `/kaggle/working/results/`. The cell builds the comparison table and a **paired patient-cluster bootstrap** of the macro-AUROC difference between the proposed model and every other experiment on each test set.

In [ ]:
res_root = WORK / 'results'
extra = [Path(p).parent for p in glob.glob(f"{CFG['INPUT_ROOT']}/**/results/*/summary.json", recursive=True)]
exp_dirs = {p.name: p for p in list(res_root.glob('*')) + extra if (p / 'summary.json').exists() and (p / 'test_predictions.npz').exists()}
print('Experiments found:', sorted(exp_dirs))
rows_cmp = []
for e, p in exp_dirs.items():
    s = json.load(open(p / 'summary.json'))['summary']
    for ts, v in s.items():
        rows_cmp.append(dict(experiment=e, test_set=ts, macro_AUROC=v['macro_AUROC'], CI_low=v['CI95'][0], CI_high=v['CI95'][1],
                             pneumonia_AUROC=v.get('pneumonia_AUROC'), macro_F1=v.get('macro_F1'), ECE_cal=v.get('ECE_cal'),
                             accuracy=v.get('accuracy'), overall_accuracy=v.get('Accuracy (overall)'),
                             overall_sensitivity=v.get('Sensitivity (overall)'), overall_precision=v.get('Precision (overall)'),
                             overall_f1=v.get('F1 (overall)'), normal_abnormal_sensitivity=v.get('Normal-vs-abnormal sensitivity')))
if rows_cmp:
    CMP = pd.DataFrame(rows_cmp); CMP.to_csv(res_root / 'comparison.csv', index=False)
    for metric in ['macro_AUROC', 'overall_accuracy', 'overall_sensitivity', 'overall_precision', 'overall_f1']:
        if CMP[metric].notna().any():
            print(f'\n{metric}'); print(CMP.pivot(index='experiment', columns='test_set', values=metric).round(4))

def paired_boot(ref, other, n_boot=1000, seed=0):
    a, b = np.load(exp_dirs[ref] / 'test_predictions.npz', allow_pickle=True), np.load(exp_dirs[other] / 'test_predictions.npz', allow_pickle=True)
    common = np.intersect1d(a['uid'], b['uid'])
    ia, ib = pd.Series(np.arange(len(a['uid'])), index=a['uid'])[common].values, pd.Series(np.arange(len(b['uid'])), index=b['uid'])[common].values
    rowsm = pd.Series(np.arange(len(meta)), index=meta.uid.values)[common].values
    out = []
    for ts in TEST_SETS:
        k = meta.test_set.values[rowsm] == ts
        if k.sum() < 20: continue
        r, pa, pb = rowsm[k], a['P_cal'][ia[k]], b['P_cal'][ib[k]]
        y, m = np.nan_to_num(Yraw[r]), (~np.isnan(Yraw[r])).astype(float)
        up, inv = np.unique(meta.patient_id.values[r], return_inverse=True); members = [np.where(inv == i)[0] for i in range(len(up))]
        rng_, d = np.random.default_rng(seed), []
        for _ in range(n_boot):
            s_ = np.concatenate([members[i] for i in rng_.integers(0, len(up), len(up))])
            d.append(macro_auc(y[s_], pa[s_], m[s_]) - macro_auc(y[s_], pb[s_], m[s_]))
        d = np.array(d); obs = macro_auc(y, pa, m) - macro_auc(y, pb, m)
        out.append(dict(reference=ref, other=other, test_set=ts, delta_macro_AUROC=obs, CI_low=np.nanpercentile(d, 2.5),
                        CI_high=np.nanpercentile(d, 97.5), p_two_sided=float(min(1, 2 * min((d <= 0).mean(), (d >= 0).mean())))))
    return out

REF = 'joint_age_adv' if 'joint_age_adv' in exp_dirs else 'joint_age'
if REF in exp_dirs and len(exp_dirs) > 1:
    PB = pd.DataFrame([r for e in exp_dirs if e != REF for r in paired_boot(REF, e, n_boot=CFG['N_BOOT'])])
    PB.to_csv(res_root / 'paired_bootstrap.csv', index=False); print(PB.round(4).to_string())

---
**Notes for the report**
* Report every number together with its test set: adult (NIH-adult), pediatric same-hospital (NIH-pediatric), pediatric other-hospital (GWCMC), and external if used.
* `macro_AUROC_without_age` in the summary shows how the proposed model behaves when the patient's age is not supplied.
* The NIH labels were mined from reports by NLP and are noisy; GWCMC is single-centre with ages 1–5. Both limitations should be stated in Chapter 6.